# PISA 2022 & HSLS:09 Educational Inequality Analysis

This notebook generates 9 interactive linked visualizations exploring educational inequality patterns using two major datasets:

- **PISA 2022**: International assessment of 613,744 students across 80+ countries, measuring mathematics, reading, and science literacy alongside comprehensive background questionnaires
- **HSLS:09**: U.S. High School Longitudinal Study tracking 23,503 students from 9th grade through postsecondary education, capturing academic trajectories and STEM pathways

## Research Themes

1. **Parental Education and Socioeconomic Influences**: How family background shapes academic outcomes and career aspirations
2. **Immigration Status and Academic Achievement**: The immigrant paradox and generational patterns in educational performance
3. **Gender Gaps Across Domains**: Persistent disparities in mathematics and reading achievement
4. **Geographic and Regional Disparities**: Spatial variation in STEM enrollment and academic trajectories
5. **Mathematics Anxiety, Self-Efficacy, and Persistence**: Psychological factors influencing achievement
6. **Technology Resources and STEM Interest**: The digital divide and its implications for career pathways

## Visualization Architecture

All visualizations employ a linked dual-panel design where selections on the **Driver Plot (Left)** filter the **Driven Plot (Right)**, enabling exploratory analysis across multiple dimensions simultaneously.

## Visualization Encoding Summary

| Viz | Title | Driver Plot (Left) | Driven Plot (Right) | Filter Field |
|-----|-------|-------------------|---------------------|--------------|
| **1** | Parental Education, Income, and STEM Aspirations | `GroupedBar` x: Parent_Ed, xOffset: Student_Expect, y: Avg_Income | `GroupedBar` y: Locale, yOffset: Gender, x: STEM_Rate | Parent_Education |
| **2** | Digital Resources, Immigration, and Academic Performance | `Line+Point` x: Parent_Ed, y: ICT_Resources, color: SES_Quartile | `Bubble` x: Home_Possessions, y: Performance, color: Immigration | SES_Quartile |
| **3** | Internet Usage and Gender Achievement Gaps | `Bar` x: Internet_Usage, y: Math_Score | `GroupedBar` x: Continent, xOffset: Subject, y: Gender_Gap | Internet_Usage_Group |
| **4** | Regional STEM Enrollment and GPA Trajectories | `USChoropleth` geo: states, color: STEM_Count | `Line` x: Grade, y: GPA, color: Race | Region |
| **5** | Socioeconomic Status and STEM Self-Efficacy | `Bubble` x: SES_z, y: Efficacy_z, color: Continent | `Density` x: STEM_Interest, color: Gender | Continent |
| **6** | Technology Resources and STEM Interest | `Bubble` x: ICT_z, y: STEM_z, color: Continent | `Density` x: ICT_Behavior, color: OECD_Status | Continent |
| **7** | Mathematics Anxiety, School Belonging, and Family Background | `Circle` x: Parent_Ed, y: Anxiety, color: SES_Tercile | `Scatter+Regression` x: Belonging, y: Anxiety, color: Gender | Parent_Education |
| **8** | Regional Academic Achievement by Domain and Gender | `Bar` x: Continent, y: Math_Score | `GroupedBar` y: Category, x: Score, color: Subject | Continent |
| **9** | School Belonging, Immigration, and Academic Outcomes | `Line` x: Domain, y: Score, color: Belonging_Level | `Scatter+Regression` x: Efficacy, y: Persistence, color: Immigration | Belonging_Level |

## Imports and Configuration

In [ ]:
import pandas as pd
import altair as alt
import numpy as np
from pathlib import Path
import json

alt.data_transformers.disable_max_rows()

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../projects/FinalProject/assets/json")
OUTPUT_DIR.mkdir(parents = True, exist_ok = True)

DARK_CONFIG = {
    "background": "#030712",
    "view": {"stroke": "transparent"},
    "axis": {
        "labelColor": "#E0E0E0",
        "titleColor": "#FFFFFF",
        "gridColor": "#333333",
        "domainColor": "#444444",
        "tickColor": "#444444"
    },
    "legend": {"labelColor": "#E0E0E0", "titleColor": "#FFFFFF"},
    "title": {"color": "#FFFFFF", "subtitleColor": "#B0B0B0"}
}

CONTINENT_MAP = {
    "USA": "North America", "CAN": "North America", "MEX": "North America", "PAN": "North America",
    "CRI": "North America", "DOM": "North America", "JAM": "North America", "PRI": "North America",
    "BRA": "South America", "ARG": "South America", "CHL": "South America", "COL": "South America",
    "PER": "South America", "URY": "South America", "ECU": "South America",
    "GBR": "Europe", "FRA": "Europe", "DEU": "Europe", "ESP": "Europe", "ITA": "Europe",
    "PRT": "Europe", "NLD": "Europe", "BEL": "Europe", "LUX": "Europe", "CHE": "Europe",
    "AUT": "Europe", "SWE": "Europe", "NOR": "Europe", "DNK": "Europe", "FIN": "Europe",
    "ISL": "Europe", "IRL": "Europe", "POL": "Europe", "CZE": "Europe", "SVK": "Europe",
    "HUN": "Europe", "SVN": "Europe", "EST": "Europe", "LVA": "Europe", "LTU": "Europe",
    "GRC": "Europe", "TUR": "Europe", "ROU": "Europe", "BGR": "Europe", "HRV": "Europe",
    "SRB": "Europe", "MNE": "Europe", "ALB": "Europe", "BIH": "Europe", "MKD": "Europe",
    "CHN": "Asia", "HKG": "Asia", "MAC": "Asia", "TWN": "Asia", "JPN": "Asia", "KOR": "Asia",
    "SGP": "Asia", "THA": "Asia", "VNM": "Asia", "IDN": "Asia", "MYS": "Asia", "PHL": "Asia",
    "KAZ": "Asia", "QAT": "Asia", "ARE": "Asia", "SAU": "Asia", "JOR": "Asia", "LBN": "Asia",
    "KWT": "Asia", "OMN": "Asia", "BHR": "Asia", "IND": "Asia", "PAK": "Asia", "BGD": "Asia",
    "LAO": "Asia", "KHM": "Asia",
    "ZAF": "Africa", "MAR": "Africa", "TUN": "Africa", "EGY": "Africa", "SEN": "Africa",
    "AUS": "Oceania", "NZL": "Oceania", "FJI": "Oceania"
}

OECD_COUNTRIES = {
    "AUS", "AUT", "BEL", "CAN", "CHE", "CHL", "COL", "CRI", "CZE", "DEU",
    "DNK", "ESP", "EST", "FIN", "FRA", "GBR", "GRC", "HUN", "IRL", "ISL",
    "ITA", "JPN", "KOR", "LTU", "LVA", "MEX", "NLD", "NOR", "NZL", "POL",
    "PRT", "SVK", "SVN", "SWE", "TUR", "USA"
}

GENDER_COLORS = {"Female": "#E91E63", "Male": "#1976D2"}
OKABE_ITO = ["#0072B2", "#D55E00", "#E69F00", "#009E73", "#CC79A7", "#F0E442", "#56B4E9"]

def save_chart(chart, filename):
    spec = json.loads(chart.to_json())
    spec["config"] = DARK_CONFIG
    with open(OUTPUT_DIR / filename, "w") as f:
        json.dump(spec, f, indent = 2)

## Data Loading

In [ ]:
pisa_df = pd.read_csv(DATA_DIR / "pisa_subset.csv", low_memory = False)
hsls_df = pd.read_csv(DATA_DIR / "hsls_subset.csv", low_memory = False)

print(f"PISA: {pisa_df.shape[0]:,} students, {pisa_df.shape[1]} variables")
print(f"HSLS: {hsls_df.shape[0]:,} students, {hsls_df.shape[1]} variables")

## Viz 1: Parental Education, Income, and STEM Aspirations

**Left Chart**: Grouped bar chart displaying average family income by parental education level, stratified by student educational expectations
**Right Chart**: Grouped horizontal bar chart showing STEM career expectations at age 30 by school locale and gender
**Interaction**: Click on a parent education level to filter the right chart

### Results Analysis

The visualization reveals a pronounced income gradient across parental education levels that reflects the fundamental mechanisms of intergenerational advantage transmission described in Bourdieu's cultural capital theory. Families where parents hold doctoral or professional degrees report average incomes approximately five to six times higher than families where parents lack a high school diploma, demonstrating how educational attainment translates directly into economic resources that can be reinvested in children's academic development. This relationship operates through multiple channels including direct financial support for educational activities, access to neighborhoods with better schools, and the capacity to provide enrichment experiences that align with school expectations.

Within each parental education category, students with the highest educational aspirations consistently come from families with the highest incomes, revealing how economic resources and academic expectations develop in tandem through processes of status socialization. This pattern aligns with expectancy value theory, which posits that students internalize achievement expectations based on their perceived likelihood of success and the value they attach to educational outcomes. Students from higher income families receive consistent messaging about the attainability and importance of advanced education, shaping their aspirations accordingly.

The right panel reveals notable gender differences in STEM career expectations that persist across all school locale types. Male students consistently report higher rates of expecting STEM careers at age 30 compared to female students, with the gap ranging from approximately 10 to 15 percentage points depending on geographic context. The gender gap appears somewhat larger in rural and town settings compared to urban areas, suggesting that exposure to diverse role models and career pathways in metropolitan environments may partially mitigate stereotype threat effects. These patterns connect to broader literature on the "leaky pipeline" phenomenon in STEM, where gender disparities emerge early and compound over time.

### Encoding Types

The left chart employs a grouped bar design that exploits the human visual system's highly accurate perception of position and length along a common aligned scale. According to Cleveland and McGill's hierarchy of graphical perception, position along a common scale ranks as the most accurate perceptual task, enabling viewers to make precise comparisons of income values both within and across parent education categories. The xOffset encoding creates side by side groupings that facilitate within category comparison of student expectation levels while maintaining the ability to track patterns across the x axis. This design applies the Gestalt principle of proximity, grouping related bars together to signal their conceptual connection.

The right chart uses horizontal bars to accommodate the longer locale and gender category labels, improving readability without requiring angled text. The yOffset encoding creates a nested factorial display where gender comparisons are embedded within each locale category. This encoding choice prioritizes within locale gender comparisons while still enabling cross locale pattern recognition through the consistent color mapping.

### Color Maps

The left chart employs a four color categorical palette (#E57373, #FFB74D, #81C784, #64B5F6) that follows a warm to cool temperature gradient mapping onto increasing educational aspirations. Red tones signal lower aspirations while blue tones indicate graduate level expectations, leveraging conventional semantic associations where cooler colors connote higher achievement. The progression through orange and green for intermediate categories maintains perceptual distinctiveness while suggesting ordinal progression. All four colors were selected to pass WCAG contrast guidelines against the dark background and remain distinguishable under common forms of color vision deficiency.

The right chart uses the standard gender color convention (#1976D2 for male, #E91E63 for female) that has become established in educational research visualization. While these associations carry cultural baggage, using conventional mappings reduces cognitive load for viewers familiar with the domain.

### Data Transformations

The parent education variable (X1PAR1EDU) is mapped from its original numeric codes to readable category labels, collapsing the original 7 point scale into 6 displayed categories by excluding the rarely occurring "don't know" response. Student educational expectations (X1STUEDEXPCT) are reduced from 11 original categories to 4 meaningful groups: high school or less, associate's degree, bachelor's degree, and graduate or professional degree. This aggregation reduces visual complexity while preserving educationally meaningful distinctions.

Family income (X1FAMINCOME) is converted from categorical bands to numeric midpoints for averaging, with values ranging from $7,500 to $250,000. This transformation enables calculation of group means while acknowledging that the resulting averages represent approximations of central tendency within income brackets. STEM expectations are derived from the X1STU30OCC_STEM1 variable, recoded as binary (0 = non STEM, 1 = STEM) for percentage calculation.

### Interactivity

Clicking on any parent education bar in the left chart filters the right chart to display STEM expectations only for students whose parents have that education level. This linked selection enables viewers to investigate whether the gender gap in STEM aspirations varies by family socioeconomic background, a research question that speaks to debates about whether STEM underrepresentation reflects structural barriers or individual preferences. Users can trace discovery pathways from observing overall income education gradients to examining how those gradients interact with gendered career aspirations across geographic contexts.

In [ ]:
v1_parent_edu_map = {
    1: "Less than HS", 1.0: "Less than HS",
    2: "HS Diploma/GED", 2.0: "HS Diploma/GED",
    3: "Associate's", 3.0: "Associate's",
    4: "Bachelor's", 4.0: "Bachelor's",
    5: "Master's", 5.0: "Master's",
    7: "Ph.D/Prof. Degree", 7.0: "Ph.D/Prof. Degree"
}
v1_student_expect_map = {
    1: "HS or Less", 1.0: "HS or Less",
    2: "HS or Less", 2.0: "HS or Less",
    3: "Associate's", 3.0: "Associate's",
    4: "Associate's", 4.0: "Associate's",
    5: "Bachelor's", 5.0: "Bachelor's",
    6: "Bachelor's", 6.0: "Bachelor's",
    7: "Graduate/Prof", 7.0: "Graduate/Prof",
    8: "Graduate/Prof", 8.0: "Graduate/Prof",
    9: "Graduate/Prof", 9.0: "Graduate/Prof",
    10: "Graduate/Prof", 10.0: "Graduate/Prof",
    11: "Unknown", 11.0: "Unknown"
}
v1_income_map = {
    1: 7500, 1.0: 7500, 2: 25000, 2.0: 25000, 3: 45000, 3.0: 45000,
    4: 65000, 4.0: 65000, 5: 85000, 5.0: 85000, 6: 105000, 6.0: 105000,
    7: 125000, 7.0: 125000, 8: 145000, 8.0: 145000, 9: 165000, 9.0: 165000,
    10: 185000, 10.0: 185000, 11: 205000, 11.0: 205000, 12: 225000, 12.0: 225000,
    13: 250000, 13.0: 250000
}
v1_locale_map = {1: "City", 1.0: "City", 2: "Suburb", 2.0: "Suburb", 3: "Town", 3.0: "Town", 4: "Rural", 4.0: "Rural"}
v1_stem_map = {0: 0, 0.0: 0, 1: 1, 1.0: 1, 2: 1, 2.0: 1, 3: 1, 3.0: 1, 4: 1, 4.0: 1, 5: 1, 5.0: 1, 6: 1, 6.0: 1}

v1_data = hsls_df[(hsls_df["X1PAR1EDU"].notna()) &
                   (hsls_df["X1FAMINCOME"].notna()) &
                   (hsls_df["X1STUEDEXPCT"].notna()) &
                   (hsls_df["X1SEX"].isin([1, 2])) &
                   (hsls_df["X1LOCALE"].notna()) &
                   (hsls_df["X1STU30OCC_STEM1"].notna())].copy()

v1_data["parent_education"] = v1_data["X1PAR1EDU"].map(v1_parent_edu_map).fillna("Unknown")
v1_data["student_ed_expect"] = v1_data["X1STUEDEXPCT"].map(v1_student_expect_map).fillna("Unknown")
v1_data["family_income_numeric"] = v1_data["X1FAMINCOME"].map(v1_income_map)
v1_data["gender"] = v1_data["X1SEX"].map({1: "Male", 1.0: "Male", 2: "Female", 2.0: "Female"})
v1_data["school_locale"] = v1_data["X1LOCALE"].map(v1_locale_map).fillna("Unknown")
v1_data["expected_stem_2009"] = v1_data["X1STU30OCC_STEM1"].map(v1_stem_map)

v1_parent_edu_order = ["Less than HS", "HS Diploma/GED", "Associate's", "Bachelor's", "Master's", "Ph.D/Prof. Degree"]
v1_expect_order = ["HS or Less", "Associate's", "Bachelor's", "Graduate/Prof"]
v1_locale_order = ["City", "Suburb", "Town", "Rural"]

v1_income_df = v1_data[
    (v1_data["parent_education"].isin(v1_parent_edu_order)) &
    (v1_data["student_ed_expect"].isin(v1_expect_order)) &
    (v1_data["family_income_numeric"].notna())
].groupby(["parent_education", "student_ed_expect"]).agg(
    avg_income = ("family_income_numeric", "mean"),
    student_count = ("family_income_numeric", "count")
).reset_index()

v1_stem_df = v1_data[
    (v1_data["school_locale"].isin(v1_locale_order)) &
    (v1_data["gender"].isin(["Male", "Female"])) &
    (v1_data["expected_stem_2009"].notna())
].groupby(["school_locale", "gender", "parent_education"]).agg(
    stem_rate = ("expected_stem_2009", "mean"),
    student_count = ("expected_stem_2009", "count")
).reset_index()

v1_expect_colors = ["#E57373", "#FFB74D", "#81C784", "#64B5F6"]
v1_gender_colors = ["#1976D2", "#E91E63"]

v1_edu_selection = alt.selection_point(fields = ["parent_education"], name = "edu_select")

v1_left_chart = alt.Chart(v1_income_df).mark_bar(
    cursor = "pointer", cornerRadiusTopRight = 3, cornerRadiusTopLeft = 3,
    stroke = "#0f172a", strokeWidth = 0.8
).encode(
    x = alt.X("parent_education:N", title = "Parent Education Level", sort = v1_parent_edu_order,
             axis = alt.Axis(labelAngle = -45, labelFontSize = 10, labelPadding = 8, titleFontSize = 14)),
    xOffset = alt.XOffset("student_ed_expect:N", sort = v1_expect_order),
    y = alt.Y("avg_income:Q", title = "Average Family Income ($)",
             axis = alt.Axis(labelFontSize = 12, titleFontSize = 14, grid = True, gridOpacity = 0.3, format = "$,.0f"),
             scale = alt.Scale(domain = [0, 180000])),
    color = alt.Color("student_ed_expect:N", title = "Student Expectations", sort = v1_expect_order,
                     scale = alt.Scale(domain = v1_expect_order, range = v1_expect_colors),
                     legend = alt.Legend(titleFontSize = 11, labelFontSize = 9, orient = "bottom",
                                        direction = "horizontal", symbolSize = 60, padding = 8,
                                        titlePadding = 8, columns = 4, offset = 10, labelLimit = 120)),
    opacity = alt.condition(v1_edu_selection, alt.value(1), alt.value(0.3)),
    tooltip = [alt.Tooltip("parent_education:N", title = "Parent Education"),
              alt.Tooltip("student_ed_expect:N", title = "Student Expectations"),
              alt.Tooltip("avg_income:Q", title = "Avg Family Income", format = "$,.0f"),
              alt.Tooltip("student_count:Q", title = "Students", format = ",d")]
).add_params(v1_edu_selection).properties(
    width = 450, height = 450,
    title = alt.TitleParams(
        text = "Association Between Parental Education and Family Income",
        subtitle = "Stratified by student educational expectations (HSLS:09)",
        fontSize = 16, subtitleFontSize = 11,
        font = "Roboto, sans-serif", anchor = "middle", fontWeight = 700,
        color = "#FFFFFF", subtitleColor = "#E0E0E0",
        offset = 10, subtitlePadding = 4
    )
)

v1_right_chart = alt.Chart(v1_stem_df).transform_filter(v1_edu_selection).mark_bar(
    cornerRadiusTopRight = 4, cornerRadiusBottomRight = 4,
    stroke = "#0f172a", strokeWidth = 0.8
).encode(
    y = alt.Y("school_locale:N", title = "School Location", sort = v1_locale_order,
             axis = alt.Axis(labelFontSize = 13, labelPadding = 8, titleFontSize = 14)),
    yOffset = alt.YOffset("gender:N", sort = ["Male", "Female"]),
    x = alt.X("mean(stem_rate):Q", title = "% Expecting STEM Career at Age 30",
             axis = alt.Axis(format = ".0%", labelFontSize = 12, titleFontSize = 14, grid = True, gridOpacity = 0.3, labelAngle = -45),
             scale = alt.Scale(domain = [0, 0.55])),
    color = alt.Color("gender:N", title = "Gender", sort = ["Male", "Female"],
                     scale = alt.Scale(domain = ["Male", "Female"], range = v1_gender_colors),
                     legend = alt.Legend(titleFontSize = 11, labelFontSize = 9, orient = "bottom",
                                        direction = "horizontal", symbolSize = 80, padding = 8,
                                        titlePadding = 8, columns = 2, offset = 10, labelLimit = 180)),
    tooltip = [alt.Tooltip("school_locale:N", title = "School Location"),
              alt.Tooltip("gender:N", title = "Gender"),
              alt.Tooltip("mean(stem_rate):Q", title = "STEM Rate", format = ".1%"),
              alt.Tooltip("sum(student_count):Q", title = "Students", format = ",d")]
).properties(
    width = 480, height = 450,
    title = alt.TitleParams(
        text = "STEM Career Expectations at Age 30 by School Locale",
        subtitle = "Gender differences across urban and rural settings",
        fontSize = 16, subtitleFontSize = 11,
        font = "Roboto, sans-serif", anchor = "middle", fontWeight = 700,
        color = "#FFFFFF", subtitleColor = "#E0E0E0",
        offset = 10, subtitlePadding = 4
    )
)

viz1 = alt.hconcat(v1_left_chart, v1_right_chart).resolve_scale(color = "independent").configure_view(
    stroke = None,
    fill = None
).properties(
    background = "transparent"
)
save_chart(viz1, "hsls_math_identity_race.json")
viz1

## Viz 2: Digital Resources, Immigration, and Academic Performance

**Left Chart**: Line chart with points showing ICT resource access by parental education level, stratified by SES quartile
**Right Chart**: Bubble chart displaying academic achievement versus home possessions index by immigration status
**Interaction**: Click on an SES quartile to filter the right chart

### Results Analysis

The visualization reveals a consistent positive relationship between parental education and ICT resource access that persists across all socioeconomic quartiles, reflecting how digital capital accumulates alongside traditional forms of educational advantage. Students from the highest SES quartile report substantially higher ICT resource indices regardless of parental education level, demonstrating that economic resources provide access to technology infrastructure even when parents themselves may not have pursued advanced education. This pattern aligns with the digital divide literature, which documents how socioeconomic disparities manifest in differential access to computers, internet connectivity, and software resources that increasingly mediate educational opportunity.

The relationship between home possessions and academic achievement follows a monotonically increasing pattern across all three immigration status categories, confirming that material resources translate into educational outcomes regardless of nativity. First generation immigrants achieve lower absolute performance levels at each home possession level compared to native born students, yet the slope of improvement appears steeper for immigrant groups, suggesting that marginal resource gains may yield proportionally larger benefits for students navigating additional linguistic and cultural challenges. This pattern connects to the immigrant paradox literature, which documents how some immigrant groups achieve outcomes exceeding what their socioeconomic circumstances would predict.

Second generation immigrants occupy an intermediate position, performing between first generation and native born students at comparable resource levels. This generational progression reflects processes of social integration and acculturation that unfold over time, as families develop familiarity with the host country's educational system and cultural expectations.

### Encoding Types

The left chart employs a connected line mark overlaid with sized point marks, leveraging the visual system's ability to perceive both the continuous trend (via line slope) and discrete values (via point position) simultaneously. The line encoding enables rapid assessment of monotonicity and rate of change across education levels, while the points provide precise anchoring for quantitative reading. Point size encodes sample count, following the convention that visual weight should correspond to statistical reliability.

The right chart uses a bubble scatter design where horizontal position encodes home possessions (continuous), vertical position encodes achievement (continuous), and area encodes sample size (continuous). This triple encoding enables efficient display of a three dimensional relationship on a two dimensional plane, though the area channel is less precisely perceived than position.

### Color Maps

The left chart employs a sequential green palette (#C8E6C9 through #1B5E20) that uses luminance variation to signal the ordinal nature of SES quartiles. Lighter greens indicate lower SES while darker greens indicate higher SES, following the perceptual convention that darker values suggest "more" of a quantity. The sequential scheme was chosen because SES quartiles represent ordered categories, unlike the categorical immigration status variable.

The right chart uses a three color categorical palette (#1E88E5 for Native, #FF9800 for Second generation, #9C27B0 for First generation) selected for maximum perceptual distinctiveness. The colors avoid red for any immigration category to prevent deficit framing associations.

### Data Transformations

The ESCS (Economic, Social, and Cultural Status) index is divided into quartiles using pandas qcut function to create four groups of approximately equal size. This quartile approach provides balanced sample sizes for comparison while acknowledging that the underlying continuous variable contains measurement error characteristic of composite indices derived from item response theory. Parental education (HISCED) is mapped from its original 0 to 8 ISCED coding to descriptive labels spanning from "None" to "Doctoral."

Home possessions (HOMEPOS) is binned into 10 decile groups to reduce overplotting while preserving the continuous character of the relationship. The achievement measure averages the first plausible values for mathematics, reading, and science, providing a comprehensive measure of academic performance while acknowledging that plausible value methodology requires appropriate variance estimation for inferential statistics.

### Interactivity

Clicking on any SES quartile line or point in the left chart filters the right bubble chart to display only students from that socioeconomic stratum. This interaction enables investigation of whether the immigration achievement relationship varies by family socioeconomic resources, addressing research questions about whether immigrant academic success depends on economic capital or operates through alternative mechanisms such as educational selectivity or cultural orientation toward achievement.

In [ ]:
v2_hisced_map = {
    0: "None", 1: "Primary", 2: "Lower Secondary",
    3: "Upper Secondary", 4: "Post-Secondary",
    5: "Short-Cycle Tertiary", 6: "Bachelor's",
    7: "Master's", 8: "Doctoral"
}
v2_edu_order = ["None", "Primary", "Lower Secondary", "Upper Secondary",
                "Post-Secondary", "Short-Cycle Tertiary", "Bachelor's", "Master's", "Doctoral"]

v2_immig_map = {1: "Native", 2: "Second-gen", 3: "First-gen"}
v2_immig_order = ["Native", "Second-gen", "First-gen"]
v2_immig_colors = ["#1E88E5", "#FF9800", "#9C27B0"]

v2_ses_order = ["Low SES", "Lower-Mid SES", "Upper-Mid SES", "High SES"]
v2_ses_colors = ["#C8E6C9", "#81C784", "#4CAF50", "#1B5E20"]

v2_data = pisa_df[(pisa_df["HOMEPOS"].notna()) &
                   (pisa_df["ESCS"].notna()) &
                   (pisa_df["HISCED"].notna()) &
                   (pisa_df["IMMIG"].notna()) &
                   (pisa_df["ICTRES"].notna()) &
                   (pisa_df["PV1MATH"].notna()) &
                   (pisa_df["PV1READ"].notna()) &
                   (pisa_df["PV1SCIE"].notna())].copy()
v2_data = v2_data[(v2_data["IMMIG"] > 0) & (v2_data["IMMIG"] <= 3)]

v2_data["Immigration_Status"] = v2_data["IMMIG"].map(v2_immig_map)
v2_data["SES_Quartile"] = pd.qcut(v2_data["ESCS"], 4, labels = v2_ses_order)
v2_data["Avg_Performance"] = (v2_data["PV1MATH"] + v2_data["PV1READ"] + v2_data["PV1SCIE"]) / 3
v2_data["Parent_Education"] = v2_data["HISCED"].map(v2_hisced_map).fillna("Unknown")

v2_homepos_bins = pd.cut(v2_data["HOMEPOS"], bins = 10)
v2_data["HOMEPOS_Bin"] = v2_homepos_bins
v2_data["HOMEPOS_Mid"] = v2_homepos_bins.apply(lambda x: x.mid if pd.notna(x) else None)

v2_ses_select = alt.selection_point(fields = ["SES_Quartile"], name = "ses_select")

v2_line_df = v2_data[v2_data["Parent_Education"].isin(v2_edu_order)].groupby(
    ["Parent_Education", "SES_Quartile"], observed = True
).agg(
    Avg_ICTRES = ("ICTRES", "mean"),
    Count = ("ICTRES", "count")
).reset_index()

v2_line_base = alt.Chart(v2_line_df).encode(
    x = alt.X("Parent_Education:N", title = "Parent Education Level",
             sort = v2_edu_order,
             axis = alt.Axis(labelAngle = -45, labelFontSize = 10, titleFontSize = 12, labelLimit = 100)),
    y = alt.Y("Avg_ICTRES:Q", title = "ICT Resources Index",
             scale = alt.Scale(domain = [-3, 4]),
             axis = alt.Axis(labelFontSize = 11, titleFontSize = 12, grid = True, gridOpacity = 0.3)),
    color = alt.Color("SES_Quartile:N", title = "SES Quartile",
                     scale = alt.Scale(domain = v2_ses_order, range = v2_ses_colors),
                     legend = alt.Legend(orient = "top", titleFontSize = 11, labelFontSize = 10,
                                        direction = "horizontal", columns = 4)),
    opacity = alt.condition(v2_ses_select, alt.value(1), alt.value(0.3))
)

v2_lines = v2_line_base.mark_line(strokeWidth = 2.5)

v2_points = v2_line_base.mark_point(filled = True, cursor = "pointer", stroke = "#FFFFFF", strokeWidth = 1).encode(
    size = alt.Size("Count:Q", title = "Sample Size",
                   scale = alt.Scale(range = [50, 400]),
                   legend = alt.Legend(orient = "bottom", titleFontSize = 10, labelFontSize = 9)),
    tooltip = [
        alt.Tooltip("Parent_Education:N", title = "Parent Education"),
        alt.Tooltip("SES_Quartile:N", title = "SES Quartile"),
        alt.Tooltip("Avg_ICTRES:Q", title = "ICT Resources", format = ".2f"),
        alt.Tooltip("Count:Q", title = "Sample Size", format = ",d")
    ]
)

v2_line_chart = alt.layer(v2_lines, v2_points).add_params(v2_ses_select).properties(
    width = 530, height = 480,
    title = alt.TitleParams(
        text = "ICT Resource Access by Parental Education Level",
        subtitle = "Disaggregated by socioeconomic status quartile (PISA 2022)",
        fontSize = 15, subtitleFontSize = 11,
        font = "Roboto, sans-serif", anchor = "middle", fontWeight = 700,
        color = "#FFFFFF", subtitleColor = "#E0E0E0",
        offset = 10, subtitlePadding = 4
    )
)

v2_bubble_df = v2_data.groupby(
    ["HOMEPOS_Mid", "Immigration_Status", "SES_Quartile"], observed = True
).agg(
    Avg_Performance = ("Avg_Performance", "mean"),
    Count = ("Avg_Performance", "count")
).reset_index()

v2_bubble_chart = alt.Chart(v2_bubble_df).transform_filter(
    v2_ses_select
).mark_circle(
    stroke = "#0f172a", strokeWidth = 0.8
).encode(
    x = alt.X("HOMEPOS_Mid:Q", title = "Home Possessions Index",
             axis = alt.Axis(labelFontSize = 11, titleFontSize = 12)),
    y = alt.Y("mean(Avg_Performance):Q", title = "Average Academic Performance",
             scale = alt.Scale(domain = [200, 550]),
             axis = alt.Axis(labelFontSize = 11, titleFontSize = 12, grid = True, gridOpacity = 0.3)),
    color = alt.Color("Immigration_Status:N", title = "Immigration Status",
                     scale = alt.Scale(domain = v2_immig_order, range = v2_immig_colors),
                     legend = alt.Legend(orient = "top", titleFontSize = 11, labelFontSize = 10,
                                        direction = "horizontal", columns = 3)),
    size = alt.Size("sum(Count):Q", title = "Sample Size",
                   scale = alt.Scale(range = [100, 800]), legend = None),
    tooltip = [
        alt.Tooltip("HOMEPOS_Mid:Q", title = "Home Possessions", format = ".2f"),
        alt.Tooltip("Immigration_Status:N", title = "Immigration Status"),
        alt.Tooltip("mean(Avg_Performance):Q", title = "Avg Performance", format = ".1f"),
        alt.Tooltip("sum(Count):Q", title = "Sample Size", format = ",d")
    ]
).properties(
    width = 530, height = 480,
    title = alt.TitleParams(
        text = "Academic Achievement by Home Possessions Index",
        subtitle = "Comparing native and immigrant student populations",
        fontSize = 15, subtitleFontSize = 11,
        font = "Roboto, sans-serif", anchor = "middle", fontWeight = 700,
        color = "#FFFFFF", subtitleColor = "#E0E0E0",
        offset = 10, subtitlePadding = 4
    )
)

viz2 = alt.hconcat(v2_line_chart, v2_bubble_chart).resolve_scale(color = "independent").configure_view(
    stroke = None,
    fill = None
).properties(
    background = "transparent"
)
save_chart(viz2, "combined_immigration.json")
viz2

## Viz 3: Internet Usage and Gender Achievement Gaps

**Left Chart**: Bar chart displaying mean mathematics scores by daily internet usage category
**Right Chart**: Grouped bar chart showing gender achievement gaps (Male minus Female) by continent and subject domain
**Interaction**: Click on an internet usage category to filter the right chart

### Results Analysis

The visualization reveals an inverted U shaped relationship between internet usage and mathematics achievement that provides empirical support for the displacement hypothesis in educational technology research. Students reporting moderate internet usage (2 to 6 hours daily) achieve the highest mathematics scores, while both minimal usage and extreme usage (10+ hours) are associated with substantially lower performance. This pattern suggests that moderate technology engagement may provide educational benefits through access to information and learning resources, while excessive usage displaces time that could otherwise be devoted to homework, sleep, or face to face learning activities.

The gender gap analysis reveals remarkably consistent patterns across continental regions, with males outperforming females in mathematics and females outperforming males in reading in nearly all geographic contexts. The mathematics gap favoring males ranges from approximately 5 to 25 points depending on the region, while the reading gap favoring females ranges from approximately 20 to 40 points. Asian countries show the largest mathematics gap favoring males, while European countries show more compressed differences. This cross cultural consistency suggests that gender achievement gaps reflect deep structural factors in how mathematics and reading are socialized rather than idiosyncratic features of particular educational systems.

The persistence of these gaps across diverse educational contexts poses challenges for explanations rooted in differential treatment or access, suggesting that expectancy effects, stereotype threat, and gendered identity formation may play central roles.

### Encoding Types

The left chart uses simple vertical bars with height encoding achievement scores, exploiting the high perceptual accuracy of length judgments along a common aligned scale. The four category grouping reduces the original continuous internet usage variable to meaningful intensity levels, enabling pattern detection without overwhelming viewers with granular distinctions.

The right chart employs grouped bars with xOffset encoding to facilitate within continent comparison of mathematics versus reading gaps. The dashed zero reference line provides a critical visual anchor, enabling viewers to immediately distinguish positive gaps (males higher) from negative gaps (females higher) without mental calculation.

### Color Maps

The left chart uses a sequential green palette (#2e7d32 through #c8e6c9) where darker greens indicate lower internet usage and lighter greens indicate higher usage. This mapping follows the intuitive association between saturation/darkness and intensity, allowing viewers to track usage levels across the chart.

The right chart uses pink (#E8A0BF) for mathematics and blue (#4169E1) for reading, deliberately avoiding the conventional pink for female and blue for male gender coding. This choice prevents confusion between the subject domain color and the computed gap direction, clarifying that the bars represent score differences rather than gender categories.

### Data Transformations

The raw internet usage variable (ST016Q01NA) is grouped from its original 11 category scale into four meaningful intensity levels: None/Low (0 to 2 hours), Moderate (2 to 6 hours), High (6 to 10 hours), and Extreme (10+ hours). This aggregation reduces noise while preserving the substantively important curvilinear pattern.

Gender gaps are computed as Male minus Female mean scores within each continent and internet usage combination, with positive values indicating male advantage and negative values indicating female advantage. Country codes are mapped to continents using the CONTINENT_MAP dictionary to enable regional aggregation across the 80+ participating countries.

### Interactivity

Clicking on an internet usage bar filters the gender gap chart to show patterns only for students in that usage category. This interaction enables investigation of whether gender gaps vary by technology engagement intensity, addressing research questions about whether heavy internet use disproportionately affects one gender's academic performance or whether the gaps remain stable across usage patterns.

In [ ]:
INTERNET_USAGE_MAP = {
    0: "None", 1: "1-2h", 2: "2-4h", 3: "4-6h", 4: "6-8h",
    5: "8-10h", 6: "10-12h", 7: "12-14h", 8: "14-16h", 9: "16+h", 10: "Extreme"
}

v3_data = pisa_df[(pisa_df["PV1MATH"].notna()) &
                   (pisa_df["PV1READ"].notna()) &
                   (pisa_df["ST004D01T"].isin([1, 2])) &
                   (pisa_df["ST016Q01NA"].notna()) &
                   (pisa_df["CNT"].notna())].copy()

v3_data["Gender"] = v3_data["ST004D01T"].map({1: "Female", 2: "Male"})
v3_data["Continent"] = v3_data["CNT"].map(CONTINENT_MAP).fillna("Other")
v3_data["Internet_Usage"] = v3_data["ST016Q01NA"].astype(int).map(INTERNET_USAGE_MAP)

INTERNET_USAGE_GROUPED = {
    "None": "None/Low (0-2h)", "1-2h": "None/Low (0-2h)",
    "2-4h": "Moderate (2-6h)", "4-6h": "Moderate (2-6h)",
    "6-8h": "High (6-10h)", "8-10h": "High (6-10h)",
    "10-12h": "Extreme (10+h)", "12-14h": "Extreme (10+h)", "14-16h": "Extreme (10+h)",
    "16+h": "Extreme (10+h)", "Extreme": "Extreme (10+h)"
}
v3_data["Internet_Usage_Group"] = v3_data["Internet_Usage"].map(INTERNET_USAGE_GROUPED)

v3_internet_agg = v3_data.groupby("Internet_Usage_Group").agg(
    Math_Score = ("PV1MATH", "mean"),
    Reading_Score = ("PV1READ", "mean"),
    Count = ("PV1MATH", "count")
).reset_index()

v3_continent_gender = v3_data.groupby(["Continent", "Gender", "Internet_Usage_Group"]).agg(
    Math_Score = ("PV1MATH", "mean"),
    Reading_Score = ("PV1READ", "mean"),
    Count = ("PV1MATH", "count")
).reset_index()

v3_math_pivot = v3_continent_gender.pivot(index = ["Continent", "Internet_Usage_Group"], columns = "Gender", values = "Math_Score").reset_index()
v3_math_pivot.columns = ["Continent", "Internet_Usage_Group", "Female_Math", "Male_Math"]
v3_math_pivot["Math_Gap"] = v3_math_pivot["Male_Math"] - v3_math_pivot["Female_Math"]

v3_read_pivot = v3_continent_gender.pivot(index = ["Continent", "Internet_Usage_Group"], columns = "Gender", values = "Reading_Score").reset_index()
v3_read_pivot.columns = ["Continent", "Internet_Usage_Group", "Female_Read", "Male_Read"]
v3_read_pivot["Reading_Gap"] = v3_read_pivot["Male_Read"] - v3_read_pivot["Female_Read"]

v3_count_pivot = v3_continent_gender.pivot(index = ["Continent", "Internet_Usage_Group"], columns = "Gender", values = "Count").reset_index()
v3_count_pivot.columns = ["Continent", "Internet_Usage_Group", "Female_N", "Male_N"]

v3_gap_df = v3_math_pivot.merge(v3_read_pivot, on = ["Continent", "Internet_Usage_Group"]).merge(v3_count_pivot, on = ["Continent", "Internet_Usage_Group"])
v3_gap_df["Total_N"] = v3_gap_df["Female_N"] + v3_gap_df["Male_N"]
v3_gap_df = v3_gap_df.dropna()

v3_continent_order = ["Europe", "Asia", "North America", "South America", "Oceania", "Africa", "Other"]
v3_group_order = ["None/Low (0-2h)", "Moderate (2-6h)", "High (6-10h)", "Extreme (10+h)"]
v3_group_colors = ["#2e7d32", "#4caf50", "#81c784", "#c8e6c9"]

v3_gap_long = v3_gap_df[["Continent", "Internet_Usage_Group", "Math_Gap", "Reading_Gap", "Total_N"]].melt(
    id_vars = ["Continent", "Internet_Usage_Group", "Total_N"],
    var_name = "Subject",
    value_name = "Gender_Gap"
)
v3_gap_long["Subject"] = v3_gap_long["Subject"].map({"Math_Gap": "Math", "Reading_Gap": "Reading"})

v3_internet_select = alt.selection_point(fields = ["Internet_Usage_Group"], name = "internet_select", empty = "all")

v3_left_chart = alt.Chart(v3_internet_agg).mark_bar(
    stroke = "#0f172a", strokeWidth = 1, cursor = "pointer"
).encode(
    x = alt.X("Internet_Usage_Group:N", title = "Daily Internet Usage",
             sort = v3_group_order,
             axis = alt.Axis(labelFontSize = 10, titleFontSize = 12, labelAngle = -45)),
    y = alt.Y("Math_Score:Q", title = "Mean Math Score",
             scale = alt.Scale(domain = [390, 450], clamp = True),
             axis = alt.Axis(labelFontSize = 11, titleFontSize = 12, grid = True, gridOpacity = 0.3)),
    color = alt.condition(
        v3_internet_select,
        alt.Color("Internet_Usage_Group:N", title = "Usage",
                 scale = alt.Scale(domain = v3_group_order, range = v3_group_colors),
                 legend = None),
        alt.value("#444444")
    ),
    tooltip = [
        alt.Tooltip("Internet_Usage_Group:N", title = "Internet Usage"),
        alt.Tooltip("Math_Score:Q", title = "Mean Math Score", format = ".1f"),
        alt.Tooltip("Reading_Score:Q", title = "Mean Reading Score", format = ".1f"),
        alt.Tooltip("Count:Q", title = "Sample Size", format = ",d")
    ]
).add_params(v3_internet_select).properties(
    width = 350, height = 420,
    title = alt.TitleParams(
        text = "Mathematics Performance by Daily Internet Usage",
        subtitle = "Mean scores across usage intensity levels (PISA 2022)",
        fontSize = 15, subtitleFontSize = 11,
        font = "Roboto, sans-serif", anchor = "middle", fontWeight = 700,
        color = "#FFFFFF", subtitleColor = "#E0E0E0",
        offset = 10, subtitlePadding = 4
    )
)

v3_right_base = alt.Chart(v3_gap_long).transform_filter(v3_internet_select)

v3_right_bars = v3_right_base.mark_bar(strokeWidth = 0.5).encode(
    x = alt.X("Continent:N", title = "Continent",
             sort = v3_continent_order,
             axis = alt.Axis(labelFontSize = 10, titleFontSize = 12, labelAngle = -30)),
    y = alt.Y("Gender_Gap:Q", title = "Gender Gap (Male - Female)",
             axis = alt.Axis(labelFontSize = 11, titleFontSize = 12, grid = True, gridOpacity = 0.3)),
    xOffset = "Subject:N",
    color = alt.Color("Subject:N", title = "Subject",
                     scale = alt.Scale(domain = ["Math", "Reading"], range = ["#E8A0BF", "#4169E1"]),
                     legend = alt.Legend(orient = "top", titleFontSize = 11, labelFontSize = 10,
                                        direction = "horizontal", symbolSize = 80)),
    tooltip = [
        alt.Tooltip("Continent:N", title = "Continent"),
        alt.Tooltip("Subject:N", title = "Subject"),
        alt.Tooltip("Gender_Gap:Q", title = "Gender Gap (M-F)", format = ".1f"),
        alt.Tooltip("Total_N:Q", title = "Sample Size", format = ",d")
    ]
).properties(
    width = 500, height = 420,
    title = alt.TitleParams(
        text = "Gender Disparities in Mathematics and Reading Achievement",
        subtitle = "Score differentials by geographic region (positive = male advantage)",
        fontSize = 15, subtitleFontSize = 11,
        font = "Roboto, sans-serif", anchor = "middle", fontWeight = 700,
        color = "#FFFFFF", subtitleColor = "#E0E0E0",
        offset = 10, subtitlePadding = 4
    )
)

v3_zero_line = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(
    color = "#FFFFFF", strokeWidth = 1.5, strokeDash = [4, 4]
).encode(y = "y:Q")

v3_right_chart = alt.layer(v3_right_bars, v3_zero_line)

viz3 = alt.hconcat(v3_left_chart, v3_right_chart).resolve_scale(color = "independent").configure_view(
    stroke = None,
    fill = None
).properties(
    background = "transparent"
)
save_chart(viz3, "pisa_gender_efficacy_dumbbell.json")
viz3

## Viz 4: Regional STEM Enrollment and GPA Trajectories

**Left Chart**: United States choropleth map displaying STEM major enrollment counts by Census region
**Right Chart**: Multi-line chart showing GPA trajectories from 9th to 12th grade by race and ethnicity
**Interaction**: Click on a region to filter the GPA trajectory chart

### Results Analysis

The choropleth map reveals substantial regional variation in STEM major production, with the South region accounting for the largest share of STEM enrollments followed by the Midwest, Northeast, and West. This distribution reflects both population concentration and the presence of major research universities and technical institutions in particular regions. The geographic clustering of STEM education opportunity connects to broader patterns of economic development and industrial concentration that shape local labor market demand for technical skills.

The GPA trajectory analysis reveals that all racial and ethnic groups show improvement in academic performance from 9th to 12th grade, suggesting that secondary schooling generally succeeds in its developmental mission regardless of demographic background. Asian students maintain the highest GPA throughout high school while demonstrating relatively flat trajectories, suggesting that high achieving students enter high school already performing near ceiling. In contrast, American Indian and Alaska Native students show the steepest improvement trajectory, with average GPA increasing by more than half a grade point from freshman to senior year.

Achievement gaps between racial groups narrow substantially over the high school years, particularly between Black, Hispanic, and White students. This convergence pattern challenges deficit narratives that attribute persistent gaps to immutable factors, instead suggesting that school environments can facilitate academic growth and reduce disparities when students remain engaged.

### Encoding Types

The left chart employs a choropleth design using the albersUsa projection, which preserves the familiar shape of the continental United States while including Alaska and Hawaii in inset positions. Geographic encoding exploits preattentive pattern recognition, enabling viewers to immediately perceive regional clustering that would be obscured in tabular format.

The right chart uses connected line marks with overlaid points to show GPA trajectories, applying the Gestalt principle of continuity to group observations within racial categories across time. The temporal x axis follows conventional left to right reading order that maps onto chronological progression.

### Color Maps

The choropleth uses an orange sequential color scheme from the Vega oranges palette, where darker orange indicates higher STEM enrollment counts. Orange was selected as a warm, attention attracting color that maintains visibility against the dark background while avoiding the evaluative connotations of red or green.

The line chart employs the Okabe Ito palette, a seven color scheme specifically designed for accessibility by individuals with color vision deficiencies. The palette provides maximum perceptual distinctiveness across common forms of deuteranopia and protanopia while maintaining aesthetic coherence.

### Data Transformations

State level observations are aggregated to Census region level (Northeast, Midwest, South, West) both to protect respondent confidentiality in the HSLS dataset and to provide meaningful geographic units for comparison. The race variable (X1RACE) is mapped from numeric codes to descriptive labels, with some categories combined (both Hispanic categories merged) to reduce the number of displayed trajectories.

GPA data is transformed from wide format (separate columns for 9th, 10th, 11th, 12th grade) to long format suitable for line chart visualization. Values of zero or negative are excluded as these indicate missing or invalid responses in the HSLS coding scheme.

### Interactivity

Clicking on any state in the map filters the GPA trajectory chart to display patterns only for students attending schools in that Census region. This geographic filtering enables investigation of whether achievement trajectories vary by regional context, addressing research questions about the role of state education policies, regional economies, and local demographic compositions in shaping academic outcomes.

In [ ]:
v4_race_map = {
    1: "Am. Indian/Alaska Native", 1.0: "Am. Indian/Alaska Native",
    2: "Asian", 2.0: "Asian",
    3: "Black/African American", 3.0: "Black/African American",
    4: "Hispanic", 4.0: "Hispanic",
    5: "Hispanic", 5.0: "Hispanic",
    6: "More than one race", 6.0: "More than one race",
    7: "Native Hawaiian/Pacific Islander", 7.0: "Native Hawaiian/Pacific Islander",
    8: "White", 8.0: "White"
}

v4_region_state_map = {
    "Northeast": [9, 23, 25, 33, 44, 50, 34, 36, 42],
    "Midwest": [17, 18, 26, 39, 55, 19, 20, 27, 29, 31, 38, 46],
    "South": [10, 11, 12, 13, 24, 37, 45, 51, 54, 1, 21, 28, 40, 47, 48, 5, 22],
    "West": [4, 8, 16, 30, 32, 35, 49, 56, 2, 6, 15, 41, 53]
}

v4_state_to_region = {}
for region, state_ids in v4_region_state_map.items():
    for state_id in state_ids:
        v4_state_to_region[state_id] = region

v4_race_order = ["White", "Black/African American", "Hispanic", "Asian", "More than one race", "Am. Indian/Alaska Native", "Native Hawaiian/Pacific Islander"]
v4_race_colors = ["#0072B2", "#D55E00", "#E69F00", "#009E73", "#CC79A7", "#F0E442", "#56B4E9"]

v4_data = hsls_df[(hsls_df["X1RACE"].notna()) &
                   (hsls_df["X3TGPA9TH"].notna()) &
                   (hsls_df["X4RFDGMJSTEM"].notna()) &
                   (hsls_df["X1REGION"].notna())].copy()
v4_data = v4_data[(v4_data["X1RACE"] > 0) &
                   (v4_data["X1REGION"] > 0) &
                   (v4_data["X4RFDGMJSTEM"].isin([0, 1]))]

v4_data["race"] = v4_data["X1RACE"].map(v4_race_map).fillna("Unknown")
v4_data["is_stem_major"] = v4_data["X4RFDGMJSTEM"].map({0: 0, 1: 1})

v4_region_mapping = {1: "Northeast", 2: "Midwest", 3: "South", 4: "West"}
v4_data["region"] = v4_data["X1REGION"].map(v4_region_mapping).fillna("Unknown")
v4_data = v4_data[(v4_data["region"] != "Unknown") & (v4_data["race"] != "Unknown")]

v4_gpa_cols = [("9th Grade", "X3TGPA9TH"), ("10th Grade", "X3TGPA10TH"),
               ("11th Grade", "X3TGPA11TH"), ("12th Grade", "X3TGPA12TH")]
v4_gpa_frames = []
for label, col in v4_gpa_cols:
    series = pd.to_numeric(v4_data[col], errors = "coerce")
    frame = pd.DataFrame({
        "region": v4_data["region"],
        "race": v4_data["race"],
        "grade": label,
        "gpa": series
    })
    v4_gpa_frames.append(frame)
v4_gpa_long = pd.concat(v4_gpa_frames, ignore_index = True)
v4_gpa_long = v4_gpa_long.dropna(subset = ["gpa"])
v4_gpa_long = v4_gpa_long[v4_gpa_long["gpa"] > 0]

v4_region_summary = v4_data.groupby("region").agg(
    stem_count = ("is_stem_major", "sum"),
    total_students = ("is_stem_major", "size"),
    stem_share = ("is_stem_major", "mean")
).reset_index()

v4_state_region_rows = []
for state_id, region in v4_state_to_region.items():
    v4_state_region_rows.append({"id": f"{state_id:02d}", "region": region})
v4_state_region_df = pd.DataFrame(v4_state_region_rows)

v4_merged_data = v4_state_region_df.merge(v4_region_summary, on = "region", how = "left")
v4_merged_data[["stem_count", "stem_share", "total_students"]] = v4_merged_data[["stem_count", "stem_share", "total_students"]].fillna(0)

v4_us_topojson_url = "https://cdn.jsdelivr.net/npm/us-atlas@3/states-10m.json"
v4_states = alt.topo_feature(v4_us_topojson_url, "states")

v4_region_select = alt.selection_point(fields = ["region"], empty = True, name = "region_select")

v4_geo_map = alt.Chart(v4_states).mark_geoshape(stroke = "white", strokeWidth = 2).encode(
    color = alt.condition(
        v4_region_select,
        alt.Color("stem_count:Q", title = "STEM Major Student Count",
                 scale = alt.Scale(scheme = "oranges", domain = [400, 1100]),
                 legend = alt.Legend(
                     format = ",d",
                     titleFontSize = 13, labelFontSize = 11,
                     orient = "bottom", direction = "horizontal",
                     gradientLength = 200, gradientThickness = 12,
                     titlePadding = 10, labelPadding = 8, padding = 12, offset = 15,
                     values = [400, 600, 800, 1000],
                     titleAnchor = "middle", legendX = 138, legendY = 410
                 )),
        alt.value("#F0F0F0")
    ),
    strokeWidth = alt.condition(v4_region_select, alt.value(3), alt.value(1.5)),
    tooltip = [
        alt.Tooltip("region:N", title = "Region"),
        alt.Tooltip("stem_count:Q", title = "STEM Major Students", format = ",d")
    ]
).transform_lookup(
    lookup = "id",
    from_ = alt.LookupData(v4_merged_data, "id", ["region", "stem_share", "stem_count", "total_students"])
).add_params(v4_region_select).project(type = "albersUsa").properties(
    width = 475, height = 400,
    title = alt.TitleParams(
        text = "Geographic Distribution of STEM Major Enrollment",
        subtitle = "U.S. Census regions, 2016 cohort (HSLS:09)",
        fontSize = 20, subtitleFontSize = 14,
        font = "Roboto, sans-serif", anchor = "middle", fontWeight = 800,
        color = "#FFFFFF", subtitleColor = "#E0E0E0",
        offset = 10, subtitlePadding = 4
    )
)

v4_gpa_line = alt.Chart(v4_gpa_long).transform_filter(v4_region_select).mark_line(
    point = alt.OverlayMarkDef(filled = True, size = 60),
    strokeWidth = 2.5
).encode(
    x = alt.X("grade:O", title = "Grade Level",
             sort = ["9th Grade", "10th Grade", "11th Grade", "12th Grade"],
             axis = alt.Axis(labelFontSize = 12, labelPadding = 10, titleFontSize = 14,
                           titleColor = "#FFFFFF", labelColor = "#E0E0E0", labelAngle = -45)),
    y = alt.Y("mean(gpa):Q", title = "Average GPA", scale = alt.Scale(domain = [1.9, 3.5]),
             axis = alt.Axis(format = ".2f", labelFontSize = 14, titleFontSize = 14,
                           titleColor = "#FFFFFF", labelColor = "#E0E0E0")),
    color = alt.Color("race:N", title = "Race/Ethnicity", sort = v4_race_order,
                     scale = alt.Scale(domain = v4_race_order, range = v4_race_colors),
                     legend = alt.Legend(titleFontSize = 11, labelFontSize = 10, orient = "bottom",
                                        direction = "horizontal", columns = 4, symbolSize = 80,
                                        padding = 8, offset = 10, titlePadding = 8, labelLimit = 150)),
    tooltip = [
        alt.Tooltip("race:N", title = "Race/Ethnicity"),
        alt.Tooltip("grade:O", title = "Grade"),
        alt.Tooltip("mean(gpa):Q", title = "Average GPA", format = ".2f"),
        alt.Tooltip("count():Q", title = "Students", format = ",d")
    ]
).properties(
    width = 475, height = 400,
    title = alt.TitleParams(
        text = "Academic Achievement Trajectories Across High School",
        subtitle = "Grade point average by race/ethnicity, grades 9 through 12",
        fontSize = 18, subtitleFontSize = 13,
        font = "Roboto, sans-serif", anchor = "middle", fontWeight = 800,
        color = "#FFFFFF", subtitleColor = "#E0E0E0",
        offset = 10, subtitlePadding = 4
    )
)

viz4 = alt.hconcat(v4_geo_map, v4_gpa_line).resolve_scale(color = "independent").configure_view(
    stroke = None,
    fill = None
).properties(
    background = "transparent"
)
save_chart(viz4, "hsls_gpa_ses_trajectory.json")
viz4

## Viz 5: Socioeconomic Status and STEM Self-Efficacy

**Left Chart**: Bubble chart displaying standardized SES versus mathematics self-efficacy by geographic region
**Right Chart**: Density plot showing STEM interest distribution by gender for selected region
**Interaction**: Click on a continent bubble to filter the density plot

### Results Analysis

The visualization reveals a strong positive correlation between socioeconomic status and mathematics self-efficacy across geographic regions, with higher SES continents clustering in the upper right quadrant while lower SES regions occupy the lower left. European and Oceanian countries demonstrate the highest combined SES and efficacy scores, while African nations show the lowest on both dimensions. This pattern aligns with Bandura's social cognitive theory, which posits that self-efficacy develops through mastery experiences, vicarious learning, and social persuasion, all of which are more readily available in resource-rich environments.

The relationship between SES and self-efficacy operates through multiple mediating mechanisms that compound across generations. Students from higher SES backgrounds receive more consistent exposure to mathematics content outside school, experience greater parental involvement in homework and tutoring, and attend schools with more qualified teachers and better instructional materials. These accumulated advantages translate into more frequent success experiences that build confidence in mathematical ability.

The density plots reveal consistent gender differences in STEM interest across all geographic regions, with male students showing distributions shifted toward higher interest levels compared to female students. The gap between genders appears more pronounced in regions with higher overall SES, suggesting that gender socialization patterns may intensify rather than diminish in more developed contexts. This finding challenges assumptions that economic development automatically reduces gender disparities in STEM orientation.

Asian regions present a notable exception to the general SES-efficacy relationship, demonstrating high mathematics self-efficacy despite moderate SES levels. This pattern may reflect cultural emphasis on effort and persistence in mathematics education, as documented in cross-cultural psychology literature comparing Eastern and Western achievement attributions.

### Encoding Types

The left chart employs a bubble scatter design where horizontal position encodes standardized SES, vertical position encodes standardized self-efficacy, and bubble area encodes sample size. This triple encoding enables efficient display of a multivariate relationship while communicating statistical reliability through visual weight. According to Cleveland and McGill's hierarchy, position along common scales ranks as the most accurately perceived visual channel, making the SES-efficacy relationship the primary comparison while sample size serves as secondary context.

The right chart uses area marks to display kernel density estimates, with the area fill creating a sense of distribution mass that corresponds intuitively to probability density. Overlapping semi-transparent areas enable direct comparison of male and female distributions without requiring vertical separation that would complicate shape comparison. The density transformation reduces individual observations to smooth distribution curves, revealing population-level patterns that would be obscured by point overplotting.

### Color Maps

The left chart uses Altair's default categorical palette for continents, providing maximum perceptual distinctiveness across geographic regions. The palette was selected to avoid semantic associations that might bias interpretation of regional differences while maintaining sufficient contrast against the dark background.

The right chart employs the standard gender color convention with pink (#E91E63) for female and blue (#1976D2) for male students. While these associations carry cultural baggage, using conventional mappings reduces cognitive load for viewers familiar with the domain and enables rapid gender identification in the overlapping density regions. Both colors were selected at saturation and luminance levels that maintain visibility at 50% opacity while preserving distinctiveness in overlapping areas.

### Data Transformations

The socioeconomic status and mathematics self-efficacy variables undergo z-score standardization within their respective source datasets (PISA and HSLS) before aggregation. This transformation addresses the fundamental challenge of comparing indices constructed with different methodologies, where raw ESCS values from PISA cannot be directly compared to X1SES values from HSLS. By converting to standard deviation units, each student's position relative to their dataset's distribution becomes comparable across sources.

The standardization formula subtracts the within-source mean and divides by the within-source standard deviation, producing variables with mean zero and standard deviation one. This approach preserves relative rankings while enabling visualization of cross-national patterns that would otherwise be confounded by scale differences.

Student-level data undergoes stratified random sampling to reduce the dataset to manageable size while maintaining proportional representation across continent-gender combinations. The sampling procedure selects up to 2,000 students per stratum, balancing computational efficiency with statistical reliability for density estimation.

### Interactivity

Clicking on any continent bubble in the left chart filters the right density plot to display STEM interest distributions only for students from that geographic region. This linked selection enables investigation of whether gender gaps in STEM interest vary by regional context, addressing research questions about cultural moderation of gender socialization effects. Users can compare the magnitude and shape of gender differences across high-SES European countries versus lower-SES African nations, exploring whether economic development amplifies or attenuates gendered interest patterns.

In [ ]:
v5_pisa = pisa_df[["CNT", "ESCS", "MATHEFF", "MATHPERS", "ST004D01T"]].copy()
v5_pisa = v5_pisa[(v5_pisa["ESCS"].notna()) &
                  (v5_pisa["MATHEFF"].notna()) &
                  (v5_pisa["MATHPERS"].notna()) &
                  (v5_pisa["ST004D01T"].isin([1, 2]))]
v5_pisa = v5_pisa.assign(source = "PISA")
v5_pisa["continent"] = v5_pisa["CNT"].map(CONTINENT_MAP).fillna("Other")

v5_hsls = hsls_df[["X1SES", "X1MTHEFF", "X1STU30OCC_STEM1", "X1SEX"]].copy()
v5_hsls = v5_hsls[(v5_hsls["X1SES"] > -1) &
                  (v5_hsls["X1MTHEFF"] > -1) &
                  (v5_hsls["X1STU30OCC_STEM1"] >= 0) &
                  (v5_hsls["X1SEX"].isin([1, 2]))]
v5_hsls = v5_hsls.assign(CNT = "USA", continent = "North America", source = "HSLS")

v5_pisa_base = v5_pisa.rename(columns = {"ESCS": "escs", "MATHEFF": "matheff"})[["continent", "escs", "matheff"]].copy()
v5_pisa_base = v5_pisa_base.dropna(subset = ["escs", "matheff", "continent"])
v5_pisa_base["z_escs"] = (v5_pisa_base["escs"] - v5_pisa_base["escs"].mean()) / v5_pisa_base["escs"].std(ddof = 0)
v5_pisa_base["z_matheff"] = (v5_pisa_base["matheff"] - v5_pisa_base["matheff"].mean()) / v5_pisa_base["matheff"].std(ddof = 0)

v5_continent_agg = (
    v5_pisa_base.groupby("continent")
    .agg(avg_escs = ("z_escs", "mean"), avg_matheff = ("z_matheff", "mean"), n = ("z_escs", "size"))
    .reset_index()
)

v5_pisa_students = (
    v5_pisa.rename(columns = {"ST004D01T": "gender"})
    .assign(
        continent = lambda d: d["CNT"].map(CONTINENT_MAP).fillna("Other"),
        source = "PISA",
        gender = lambda d: d["gender"].map({1: "Female", 2: "Male"}),
        stem_interest = lambda d: d["MATHPERS"],
    )
    [["continent", "gender", "stem_interest", "source"]]
    .dropna(subset = ["stem_interest", "gender", "continent"])
)
v5_hsls_students = (
    v5_hsls.rename(columns = {"X1SEX": "gender"})
    .assign(
        source = "HSLS",
        gender = lambda d: d["gender"].map({1: "Male", 2: "Female"}),
        stem_interest = lambda d: d["X1STU30OCC_STEM1"],
    )
    [["continent", "gender", "stem_interest", "source"]]
    .dropna(subset = ["stem_interest", "gender", "continent"])
)
v5_hsls_students = v5_hsls_students[v5_hsls_students["stem_interest"].isin([0, 1, 2, 3, 4, 5, 6])]
v5_students = pd.concat([v5_pisa_students, v5_hsls_students], ignore_index = True)

v5_sampled_students = v5_students.groupby(["continent", "gender"]).apply(
    lambda x: x.sample(n = min(len(x), 2000), random_state = 42)
).reset_index(drop = True)

v5_continent_select = alt.selection_point(
    fields = ["continent"],
    name = "continent_select",
    empty = True
)

v5_left_chart = (
    alt.Chart(v5_continent_agg)
    .mark_circle(cursor = "pointer")
    .encode(
        x = alt.X("avg_escs:Q", title = "Mean SES (z within source)"),
        y = alt.Y("avg_matheff:Q", title = "Mean Math Self-Efficacy (z within source)"),
        size = alt.Size("n:Q", scale = alt.Scale(range = [60, 500]), legend = None),
        color = alt.Color("continent:N", title = "Continent/Region"),
        opacity = alt.condition(v5_continent_select, alt.value(1), alt.value(0.4)),
        tooltip = [
            alt.Tooltip("continent:N", title = "Continent"),
            alt.Tooltip("avg_escs:Q", title = "Mean SES", format = ".2f"),
            alt.Tooltip("avg_matheff:Q", title = "Mean Math Efficacy", format = ".2f"),
            alt.Tooltip("n:Q", title = "Sample Size", format = ",d")
        ],
    )
    .add_params(v5_continent_select)
    .properties(
        name = "view_1",
        title = {"text": "Socioeconomic Status and Mathematics Self-Efficacy", "subtitle": "Standardized mean scores by geographic region",
                "color": "#FFFFFF", "fontSize": 14, "subtitleColor": "#E0E0E0"},
        width = 280, height = 240
    )
)

v5_right_chart = (
    alt.Chart(v5_sampled_students)
    .transform_filter(v5_continent_select)
    .transform_density(
        density = "stem_interest",
        groupby = ["gender"],
        as_ = ["stem_interest", "density"],
    )
    .mark_area(opacity = 0.5)
    .encode(
        x = alt.X("stem_interest:Q", title = "STEM Interest Index"),
        y = alt.Y("density:Q", title = "Density"),
        color = alt.Color("gender:N", title = "Gender", scale = alt.Scale(domain = ["Female", "Male"], range = ["#E91E63", "#1976D2"])),
    )
    .properties(
        title = {"text": "Distribution of STEM Interest by Gender", "subtitle": "Density estimates for selected geographic region",
                "color": "#FFFFFF", "fontSize": 14, "subtitleColor": "#E0E0E0"},
        width = 280, height = 240
    )
)

viz5 = alt.hconcat(v5_left_chart, v5_right_chart).resolve_scale(color = "independent").configure_view(
    stroke = None,
    fill = None
).properties(
    background = "transparent"
)
save_chart(viz5, "combined_ses_achievement.json")
viz5

## Viz 6: Technology Resources and STEM Interest

**Left Chart**: Bubble chart displaying standardized ICT resources versus STEM interest by geographic region
**Right Chart**: Density plot showing ICT behavior distribution by OECD membership status for selected region
**Interaction**: Click on a continent bubble to filter the density plot

### Results Analysis

The visualization reveals a positive but modest relationship between technology resource access and STEM interest across geographic regions, with considerable variation in how different continents position themselves along both dimensions. European and North American regions demonstrate the highest ICT resource access while showing moderate STEM interest levels, suggesting that technology availability alone does not translate directly into heightened interest in STEM fields. This pattern challenges techno-deterministic narratives that assume digital access automatically generates STEM engagement.

Asian regions present a particularly instructive case, demonstrating relatively high STEM interest despite moderate ICT resource indices. Countries like Japan, Korea, and Singapore have built educational cultures that emphasize mathematics and science achievement independent of digital technology saturation. This suggests that pedagogical approaches and cultural values may exert stronger influence on STEM orientation than raw technology access.

The OECD versus non-OECD comparison in the density plots reveals systematic differences in ICT behavior patterns that persist across geographic regions. OECD member countries show ICT behavior distributions shifted toward higher engagement levels, with narrower variance indicating more uniform technology use patterns. Non-OECD countries display wider distributions with substantial mass at lower engagement levels, reflecting the digital divide in both access and usage intensity.

The distinction between ICT resources and ICT behavior proves analytically important. Resources measure availability of devices and connectivity, while behavior captures actual usage patterns for educational and informational purposes. The gap between these constructs is often larger in developing contexts where devices may be available but consistent educational use remains limited by factors including connectivity costs, digital literacy, and curricular integration.

### Encoding Types

The left chart employs a bubble scatter design identical in structure to Visualization 5, with horizontal position encoding standardized ICT resources, vertical position encoding standardized STEM interest, and bubble area encoding sample size. The consistent visual grammar across visualizations enables viewers to build familiarity with the encoding scheme while focusing on substantive differences in the data.

The right chart uses area marks to display kernel density estimates with explicit bandwidth specification (0.25) to control smoothing level. The bandwidth parameter balances the competing goals of revealing distribution shape while suppressing noise from sampling variation. Semi-transparent fills enable comparison of overlapping OECD and non-OECD distributions without requiring spatial separation.

### Color Maps

The left chart uses Altair's default categorical palette for continents, maintaining consistency with Visualization 5 to enable cross-visualization comparison of regional patterns.

The right chart uses green (#4CAF50) for OECD countries and orange (#FF9800) for non-OECD countries. These colors were selected for maximum distinctiveness while avoiding red-green combinations that pose challenges for individuals with common forms of color vision deficiency. Green for OECD and orange for non-OECD follows the conventional association of green with "developed" status while avoiding deficit-framing through pejorative color choices for developing nations.

### Data Transformations

ICT resource and STEM interest variables undergo within-source z-score standardization using grouped transformations that calculate means and standard deviations separately for PISA and HSLS observations. This approach preserves the relative structure within each dataset while enabling cross-source visualization on common scales. The standardization formula applies to grouped data, ensuring that PISA students are compared to the PISA distribution and HSLS students to the HSLS distribution.

ICT behavior variables are derived from questionnaire items capturing technology usage for educational and informational purposes. Missing value codes (-9, -8, -7, -5) are excluded to prevent contamination of density estimates with structurally missing data. The remaining continuous values are passed to the kernel density estimator with explicit bandwidth specification.

The SES tercile variable divides the ESCS (PISA) and X1SES (HSLS) distributions into three equal-sized groups (Low, Mid, High) using quantile-based binning. This categorical transformation enables OECD status comparisons within SES strata, though the density visualization focuses on the simpler OECD binary classification.

### Interactivity

Clicking on any continent bubble in the left chart filters the right density plot to display ICT behavior distributions only for students from that geographic region. This interaction enables investigation of whether the OECD digital divide varies by regional context, addressing questions about whether technology usage gaps between developed and developing nations differ across continents with distinct economic and educational characteristics.

In [ ]:
v6_pisa = pisa_df[["CNT", "ICTRES", "MATHPERS", "ST004D01T", "ESCS"]].copy()
v6_pisa = v6_pisa.dropna(subset = ["ICTRES", "MATHPERS", "CNT"])
v6_pisa = v6_pisa.assign(
    source = "PISA",
    continent = lambda d: d["CNT"].map(CONTINENT_MAP).fillna("Other"),
    gender = lambda d: d["ST004D01T"].map({1: "Female", 2: "Male"}),
    ict_resource = lambda d: d["ICTRES"],
    stem_interest = lambda d: d["MATHPERS"],
    OECD_Status = lambda d: d["CNT"].apply(lambda x: "OECD" if x in OECD_COUNTRIES else "Non-OECD")
)

v6_hsls = hsls_df[["X1SES", "X1MTHINT", "X1SEX"]].copy()
v6_hsls = v6_hsls[(v6_hsls["X1SES"].notna()) & (v6_hsls["X1MTHINT"].notna())]
v6_hsls = v6_hsls.assign(
    CNT = "USA",
    source = "HSLS",
    continent = "North America",
    gender = lambda d: d["X1SEX"].map({1: "Male", 2: "Female"}),
    ict_resource = lambda d: d["X1SES"],
    stem_interest = lambda d: d["X1MTHINT"],
    OECD_Status = "OECD"
)

v6_left_base = pd.concat([
    v6_pisa[["continent", "ict_resource", "stem_interest", "source"]],
    v6_hsls[["continent", "ict_resource", "stem_interest", "source"]],
], ignore_index = True)
v6_left_base = v6_left_base.dropna(subset = ["ict_resource", "stem_interest", "continent", "source"])
v6_left_base["z_resource"] = v6_left_base.groupby("source")["ict_resource"].transform(lambda x: (x - x.mean()) / x.std(ddof = 0))
v6_left_base["z_stem"] = v6_left_base.groupby("source")["stem_interest"].transform(lambda x: (x - x.mean()) / x.std(ddof = 0))

v6_continent_df = (
    v6_left_base.groupby("continent")
    .agg(avg_res = ("z_resource", "mean"), avg_stem = ("z_stem", "mean"), n = ("z_resource", "size"))
    .reset_index()
)

v6_pisa_students = v6_pisa[["continent", "OECD_Status", "stem_interest", "source"]].dropna()
v6_hsls_students = v6_hsls[["continent", "OECD_Status", "stem_interest", "source"]].dropna()
v6_students_df = pd.concat([v6_pisa_students, v6_hsls_students], ignore_index = True)

v6_students_df = (
    v6_students_df.groupby(["continent", "OECD_Status"], group_keys = False)
    .apply(lambda x: x.sample(n = min(500, len(x)), random_state = 42))
    .reset_index(drop = True)
)

v6_continent_select = alt.selection_point(fields = ["continent"], name = "v6_continent_select")

v6_left_chart = (
    alt.Chart(v6_continent_df)
    .mark_circle(size = 150, cursor = "pointer")
    .encode(
        x = alt.X("avg_res:Q", title = "Mean ICT Resources (z within source)", scale = alt.Scale(domain = [-1.0, 1.0])),
        y = alt.Y("avg_stem:Q", title = "Mean STEM Interest (z within source)", scale = alt.Scale(domain = [-0.4, 0.2])),
        color = alt.Color("continent:N", title = "Continent/Region"),
        opacity = alt.condition(v6_continent_select, alt.value(1), alt.value(0.5)),
        tooltip = [
            alt.Tooltip("continent:N", title = "Continent"),
            alt.Tooltip("avg_res:Q", title = "Mean ICT Resources", format = ".2f"),
            alt.Tooltip("avg_stem:Q", title = "Mean STEM Interest", format = ".2f"),
            alt.Tooltip("n:Q", title = "Sample Size", format = ",d")
        ],
    )
    .add_params(v6_continent_select)
    .properties(
        title = {"text": "Relationship Between ICT Resources and STEM Interest", "subtitle": "Standardized indices by geographic region",
               "color": "#FFFFFF", "fontSize": 14, "subtitleColor": "#E0E0E0"},
        width = 450, height = 400
    )
)

v6_right_chart = (
    alt.Chart(v6_students_df)
    .transform_filter(v6_continent_select)
    .transform_density(
        density = "stem_interest",
        groupby = ["OECD_Status"],
        as_ = ["stem_interest", "density"],
        bandwidth = 0.25,
    )
    .mark_area(opacity = 0.5)
    .encode(
        x = alt.X("stem_interest:Q", title = "STEM Interest Index"),
        y = alt.Y("density:Q", title = "Density"),
        color = alt.Color("OECD_Status:N",
                       scale = alt.Scale(domain = ["OECD", "Non-OECD"], range = ["#4CAF50", "#FF9800"]),
                       title = "OECD Status"),
        tooltip = [
            alt.Tooltip("OECD_Status:N", title = "OECD Status"),
            alt.Tooltip("stem_interest:Q", title = "STEM Interest", format = ".2f"),
            alt.Tooltip("density:Q", title = "Density", format = ".3f")
        ],
    )
    .properties(
        title = {"text": "STEM Interest Distribution: OECD vs Non-OECD Countries", "subtitle": "Kernel density estimates by development status",
               "color": "#FFFFFF", "fontSize": 14, "subtitleColor": "#E0E0E0"},
        width = 400, height = 400
    )
)

viz6 = alt.hconcat(v6_left_chart, v6_right_chart).resolve_scale(color = "independent").configure_view(
    stroke = None,
    fill = None
).properties(
    background = "transparent"
)
save_chart(viz6, "combined_parent_education.json")
viz6

## Viz 7: Mathematics Anxiety, School Belonging, and Family Background

**Left Chart**: Circle chart displaying mean mathematics anxiety by parental education level, stratified by SES tercile
**Right Chart**: Scatter plot with regression lines showing school belonging versus math anxiety by gender
**Interaction**: Click on a parental education level to filter the scatter plot

### Results Analysis

The visualization reveals a consistent negative gradient between parental education and mathematics anxiety, with students whose parents hold bachelor's degrees or higher reporting substantially lower anxiety than students whose parents lack formal education. This pattern aligns with social reproduction theory, which posits that educational advantages transmit across generations through multiple mechanisms including cognitive stimulation, academic socialization, and stress buffering. Parents with higher education levels may provide more effective support for mathematics learning, reducing the uncertainty and frustration that generate anxiety.

Within each parental education category, SES terciles reveal additional stratification in anxiety levels. Low SES students consistently report higher anxiety than high SES students even when their parents have similar educational attainment, suggesting that economic resources exert independent effects beyond the cultural capital associated with parental education. This finding supports multidimensional conceptualizations of socioeconomic status that distinguish between different forms of family capital.

The scatter plot reveals a negative correlation between school belonging and mathematics anxiety that persists across gender groups, confirming that students who feel connected to their school community experience less academic stress. The protective effect of belonging operates through multiple pathways including social support from peers and teachers, reduced isolation and alienation, and stronger identification with academic goals.

Gender differences in both the level and slope of the belonging-anxiety relationship emerge clearly from the visualization. Female students report higher average anxiety levels across all belonging scores, consistent with research documenting gender gaps in mathematics affect that begin in elementary school and persist through secondary education. The regression lines suggest similar slopes for both genders, indicating that the protective effect of belonging operates equivalently for male and female students even though females start from a higher anxiety baseline.

### Encoding Types

The left chart employs circle marks positioned along two dimensions, with horizontal position encoding parental education category and vertical position encoding mean anxiety. Circle color encodes SES tercile, creating a grouped display that reveals both education effects and within-education SES stratification. The circle size remains constant to emphasize the positional and color encodings that carry the primary analytical content.

The right chart combines scatter point marks with regression line overlays, enabling viewers to perceive both individual-level variation and group-level trends. The scatter points reveal the joint distribution of belonging and anxiety, while the regression lines summarize the linear relationship within each gender group. This layered approach follows the principle of progressive disclosure, allowing viewers to first perceive overall patterns before examining individual observations.

### Color Maps

The left chart uses a three-color palette (#E57373 red, #FFB74D orange, #64B5F6 blue) for SES terciles that follows a warm-to-cool temperature gradient. Red for low SES and blue for high SES leverages semantic associations where warmer colors suggest disadvantage or concern while cooler colors suggest advantage or calm. The orange intermediate category provides clear perceptual separation between the extremes.

The right chart employs the standard gender color convention with pink (#E91E63) for female and blue (#1976D2) for male students. Maintaining consistent gender color mapping across visualizations reduces cognitive load and enables rapid cross-visualization pattern recognition. Both colors maintain visibility at the 50% opacity used for scatter points while providing clear differentiation in the regression line overlays.

### Data Transformations

The parental education variable (HISCED) is mapped from its original ISCED coding scheme to six consolidated categories that represent meaningful educational transitions: No Education, Primary, Lower Secondary, Upper Secondary, Post-Secondary, and Bachelor's or higher. This aggregation reduces visual complexity while preserving the substantive educational gradients that shape student outcomes.

The ESCS (Economic, Social, and Cultural Status) index is divided into terciles using quantile-based binning, creating three groups of approximately equal size. This transformation converts the continuous composite index into categorical levels that can be displayed as discrete color categories while maintaining the ordinal relationship between groups.

The scatter plot data undergoes random sampling to reduce the display to 2,000 observations, preventing overplotting that would obscure distributional patterns. The regression transformation fits linear models within each gender group, extracting trend lines that summarize the belonging-anxiety relationship without displaying the full regression statistics.

### Interactivity

Clicking on any parental education category in the left chart filters the right scatter plot to display only students whose parents have that education level. This linked selection enables investigation of whether the belonging-anxiety relationship varies by family educational background, addressing questions about whether the protective effect of school connectedness operates differently across socioeconomic contexts. Users can explore whether belonging provides stronger buffering against anxiety for students from disadvantaged backgrounds who may lack alternative sources of academic support.

In [ ]:
v7_parent_edu_map = {
    0: "No Education", 0.0: "No Education",
    1: "Primary", 1.0: "Primary", 2: "Primary", 2.0: "Primary",
    3: "Lower Secondary", 3.0: "Lower Secondary", 4: "Lower Secondary", 4.0: "Lower Secondary",
    5: "Upper Secondary", 5.0: "Upper Secondary", 6: "Upper Secondary", 6.0: "Upper Secondary",
    7: "Post-Secondary", 7.0: "Post-Secondary", 8: "Post-Secondary", 8.0: "Post-Secondary",
    9: "Bachelor's+", 9.0: "Bachelor's+", 10: "Bachelor's+", 10.0: "Bachelor's+"
}

v7_gender_map = {1: "Female", 1.0: "Female", 2: "Male", 2.0: "Male"}

v7_data = pisa_df[
    (pisa_df["HISCED"].notna()) &
    (pisa_df["ESCS"].notna()) &
    (pisa_df["ANXMAT"].notna()) &
    (pisa_df["BELONG"].notna()) &
    (pisa_df["ST004D01T"].isin([1, 2, 1.0, 2.0]))
].copy()

v7_data["parent_education"] = v7_data["HISCED"].map(v7_parent_edu_map).fillna("Unknown")
v7_data["gender"] = v7_data["ST004D01T"].map(v7_gender_map)

v7_escs_terciles = v7_data["ESCS"].quantile([0.33, 0.67]).values
v7_data["ses_level"] = pd.cut(
    v7_data["ESCS"],
    bins = [-np.inf, v7_escs_terciles[0], v7_escs_terciles[1], np.inf],
    labels = ["Low SES", "Medium SES", "High SES"]
)

v7_parent_edu_order = ["No Education", "Primary", "Lower Secondary", "Upper Secondary", "Post-Secondary", "Bachelor's+"]
v7_ses_order = ["Low SES", "Medium SES", "High SES"]
v7_gender_colors = ["#E91E63", "#1976D2"]
v7_ses_colors = ["#E57373", "#FFB74D", "#64B5F6"]

v7_heatmap_df = (
    v7_data[v7_data["parent_education"] != "Unknown"]
    .groupby(["parent_education", "ses_level"], observed = True)
    .agg(
        avg_anxiety = ("ANXMAT", "mean"),
        student_count = ("ANXMAT", "count")
    )
    .reset_index()
)

v7_scatter_df = v7_data[["parent_education", "gender", "BELONG", "ANXMAT"]].dropna().copy()
v7_scatter_df = v7_scatter_df[v7_scatter_df["parent_education"] != "Unknown"]
v7_scatter_df = v7_scatter_df.sample(n = min(2000, len(v7_scatter_df)), random_state = 42)

v7_edu_selection = alt.selection_point(fields = ["parent_education"], name = "v7_edu_select")

v7_left_chart = (
    alt.Chart(v7_heatmap_df)
    .mark_circle(cursor = "pointer", stroke = "#0f172a", strokeWidth = 1, size = 200)
    .encode(
        x = alt.X("parent_education:N", title = "Parental Education Level",
                sort = v7_parent_edu_order,
                axis = alt.Axis(labelAngle = -45, labelFontSize = 10, labelPadding = 8, titleFontSize = 14)),
        y = alt.Y("avg_anxiety:Q", title = "Mean Math Anxiety",
                axis = alt.Axis(labelFontSize = 12, titleFontSize = 14, grid = True, gridOpacity = 0.3),
                scale = alt.Scale(domain = [-3, 1])),
        color = alt.Color("ses_level:N", title = "SES Level", sort = v7_ses_order,
                       scale = alt.Scale(domain = v7_ses_order, range = v7_ses_colors),
                       legend = alt.Legend(titleFontSize = 11, labelFontSize = 9, orient = "bottom",
                                        direction = "horizontal", symbolSize = 80, padding = 8)),
        opacity = alt.condition(v7_edu_selection, alt.value(0.9), alt.value(0.3)),
        tooltip = [
            alt.Tooltip("parent_education:N", title = "Parent Education"),
            alt.Tooltip("ses_level:N", title = "SES Level"),
            alt.Tooltip("avg_anxiety:Q", title = "Mean Anxiety", format = ".3f"),
            alt.Tooltip("student_count:Q", title = "Students", format = ",d")
        ]
    )
    .add_params(v7_edu_selection)
    .properties(
        width = 450, height = 400,
        title = alt.TitleParams(
            text = "Mathematics Anxiety by Parental Education and SES",
            subtitle = "Mean anxiety scores stratified by socioeconomic tercile",
            fontSize = 16, subtitleFontSize = 11,
            font = "Roboto, sans-serif", anchor = "middle", fontWeight = 700,
            color = "#FFFFFF", subtitleColor = "#E0E0E0",
            offset = 10, subtitlePadding = 4
        )
    )
)

v7_right_scatter = (
    alt.Chart(v7_scatter_df)
    .transform_filter(v7_edu_selection)
    .mark_circle(size = 30, opacity = 0.5)
    .encode(
        x = alt.X("BELONG:Q", title = "School Belonging Score",
                scale = alt.Scale(domain = [-4, 4])),
        y = alt.Y("ANXMAT:Q", title = "Math Anxiety Score",
                scale = alt.Scale(domain = [-3, 3])),
        color = alt.Color("gender:N", title = "Gender",
                       scale = alt.Scale(domain = ["Female", "Male"], range = v7_gender_colors),
                       legend = alt.Legend(orient = "top")),
        tooltip = [
            alt.Tooltip("gender:N", title = "Gender"),
            alt.Tooltip("BELONG:Q", title = "School Belonging", format = ".2f"),
            alt.Tooltip("ANXMAT:Q", title = "Math Anxiety", format = ".2f")
        ]
    )
)

v7_right_regression = (
    alt.Chart(v7_scatter_df)
    .transform_filter(v7_edu_selection)
    .transform_regression("BELONG", "ANXMAT", groupby = ["gender"])
    .mark_line(strokeWidth = 3)
    .encode(
        x = alt.X("BELONG:Q"),
        y = alt.Y("ANXMAT:Q"),
        color = alt.Color("gender:N", scale = alt.Scale(domain = ["Female", "Male"], range = v7_gender_colors))
    )
)

v7_right_chart = alt.layer(v7_right_scatter, v7_right_regression).properties(
    title = {"text": "School Belonging and Mathematics Anxiety",
           "subtitle": "Gender-specific associations with regression trends",
           "color": "#FFFFFF", "fontSize": 14, "subtitleColor": "#E0E0E0"},
    width = 400, height = 400
)

viz7 = alt.hconcat(v7_left_chart, v7_right_chart).resolve_scale(color = "independent").configure_view(
    stroke = None,
    fill = None
).properties(
    background = "transparent"
)
save_chart(viz7, "pisa_anxiety_performance_heatmap.json")
viz7

## Viz 8: Regional Academic Achievement by Domain and Gender

**Left Chart**: Bar chart displaying mean mathematics scores by geographic region
**Right Chart**: Horizontal grouped bar chart showing academic performance across math, science, and reading by gender
**Interaction**: Click on a continent to filter the gender performance breakdown

### Results Analysis

The visualization reveals substantial variation in mathematics achievement across geographic regions, with Asian and European countries consistently outperforming other continents. The 150-point gap between top and bottom performing regions represents approximately 1.5 standard deviations on the PISA scale, equivalent to roughly four years of schooling. This magnitude of regional disparity reflects fundamental differences in educational system quality, resource allocation, and cultural orientation toward academic achievement.

Within regions, the gender-by-subject breakdown reveals the familiar pattern of male advantage in mathematics and female advantage in reading that has persisted across PISA administrations since 2000. The mathematics gap favoring males averages approximately 10-15 points across regions, while the reading gap favoring females averages 30-40 points. Science achievement shows more compressed gender differences, with neither gender demonstrating consistent advantage across all regions.

The interaction between region and gender reveals important heterogeneity in these patterns. Asian countries show smaller mathematics gender gaps than European or American regions, suggesting that cultural expectations and educational practices moderate the relationship between gender and mathematics achievement. Some Nordic European countries have achieved near parity in mathematics while maintaining or extending female reading advantages, demonstrating that gender gaps are malleable rather than fixed.

The consistency of female reading advantages across all regions presents a puzzle for theories emphasizing structural barriers to female achievement. If educational systems systematically disadvantaged girls, one would expect deficits across all domains rather than selective patterns favoring different genders in different subjects. This domain-specific pattern suggests that socialization processes and stereotype threat effects operate differently across academic contexts.

### Encoding Types

The left chart uses vertical bars with height encoding mean mathematics scores, the most precisely perceived visual channel according to Cleveland and McGill's hierarchy. Color encoding reinforces continental grouping while enabling rapid visual identification across the legend. The scale domain (350-550) was selected to maximize visual discrimination in the range where most observations fall while maintaining the PISA anchor at 500.

The right chart uses horizontal bars with the Gender × Subject categorical combination on the y-axis, enabling comparison across six categories without requiring angled labels. The horizontal orientation accommodates longer category labels while maintaining readable text. The color encoding by subject enables viewers to track patterns across gender groups, comparing all mathematics bars or all reading bars simultaneously.

### Color Maps

The left chart uses a seven-color categorical palette for continents, with colors selected to avoid semantic associations while maintaining maximum perceptual distinctiveness. Green for Europe, orange for Asia, and blue for North America follow conventional geographic color associations used in cartography, facilitating recognition without requiring constant legend reference.

The right chart uses pink (#E91E63) for mathematics, cyan (#00BCD4) for science, and green (#4CAF50) for reading. This assignment deliberately avoids using pink for female-associated content or blue for male-associated content, preventing confusion between subject color and gender categories. The three colors maintain high contrast against both dark backgrounds and each other in the grouped display.

### Data Transformations

Achievement scores are computed by averaging all available plausible values (PV1 through PV10) for each subject domain. This averaging approach provides more stable estimates than using a single plausible value while acknowledging that proper variance estimation for inferential statistics would require Rubin's combining rules. For visualization purposes, the averaged plausible values provide representative point estimates suitable for between-group comparison.

The reshaping transformation converts the wide format (separate columns for math, science, reading scores) to long format suitable for faceted visualization. The Category variable concatenates gender and subject labels to create six unique row positions in the horizontal bar chart, enabling the grouped comparison structure.

Regional aggregation maps individual country codes to continent-level categories using the CONTINENT_MAP dictionary. This aggregation enables cross-cultural comparison at a manageable granularity while acknowledging that within-continent variation can be substantial, particularly in geographically and culturally diverse regions like Asia.

### Interactivity

Clicking on any continent bar in the left chart filters the right chart to display gender-by-subject patterns only for students from that region. This linked selection enables investigation of whether gender achievement patterns vary systematically across cultural and educational contexts. Users can compare the magnitude and direction of gender gaps across high-performing Asian countries, egalitarian Nordic nations, and developing world contexts, exploring the cultural contingency of gender-achievement relationships.

In [ ]:
v8_pisa = pisa_df[
    (pisa_df["ESCS"].notna()) &
    (pisa_df["PV1MATH"].notna()) &
    (pisa_df["PV1SCIE"].notna()) &
    (pisa_df["PV1READ"].notna()) &
    (pisa_df["CNT"].notna()) &
    (pisa_df["ST004D01T"].isin([1, 2]))
].copy()

v8_math_cols = [f"PV{i}MATH" for i in range(1, 11)]
v8_scie_cols = [f"PV{i}SCIE" for i in range(1, 11)]
v8_read_cols = [f"PV{i}READ" for i in range(1, 11)]
v8_available_math = [col for col in v8_math_cols if col in v8_pisa.columns]
v8_available_scie = [col for col in v8_scie_cols if col in v8_pisa.columns]
v8_available_read = [col for col in v8_read_cols if col in v8_pisa.columns]

v8_pisa["Math_Score"] = v8_pisa[v8_available_math].mean(axis = 1)
v8_pisa["Science_Score"] = v8_pisa[v8_available_scie].mean(axis = 1)
v8_pisa["Reading_Score"] = v8_pisa[v8_available_read].mean(axis = 1)
v8_pisa["Gender"] = v8_pisa["ST004D01T"].map({1: "Female", 2: "Male"})
v8_pisa["Continent"] = v8_pisa["CNT"].map(CONTINENT_MAP).fillna("Other")

v8_continent_agg = v8_pisa.groupby(["Continent"]).agg(
    Mean_Math = ("Math_Score", "mean"),
    Mean_Science = ("Science_Score", "mean"),
    Mean_Reading = ("Reading_Score", "mean"),
    n = ("Continent", "size")
).reset_index()

v8_continent_gender_agg = v8_pisa.groupby(["Continent", "Gender"]).agg(
    Mean_Math = ("Math_Score", "mean"),
    Mean_Science = ("Science_Score", "mean"),
    Mean_Reading = ("Reading_Score", "mean"),
    n = ("Continent", "size")
).reset_index()

v8_gender_subject_long = v8_continent_gender_agg.melt(
    id_vars = ["Continent", "Gender", "n"],
    value_vars = ["Mean_Math", "Mean_Science", "Mean_Reading"],
    var_name = "Subject",
    value_name = "Score"
)
v8_gender_subject_long["Subject"] = v8_gender_subject_long["Subject"].map({"Mean_Math": "Math", "Mean_Science": "Science", "Mean_Reading": "Reading"})
v8_gender_subject_long["Category"] = v8_gender_subject_long["Gender"] + " " + v8_gender_subject_long["Subject"]

v8_continent_select = alt.selection_point(fields = ["Continent"], name = "continent_select", empty = "all")

v8_continent_order = ["Europe", "Asia", "North America", "South America", "Oceania", "Africa", "Other"]
v8_continent_colors = ["#4CAF50", "#FF9800", "#2196F3", "#9C27B0", "#00BCD4", "#FF5722", "#607D8B"]
v8_subject_colors = ["#E91E63", "#00BCD4", "#4CAF50"]
v8_category_order = ["Female Math", "Female Science", "Female Reading", "Male Math", "Male Science", "Male Reading"]

v8_left_chart = (
    alt.Chart(v8_continent_agg)
    .mark_bar(cursor = "pointer", cornerRadiusTopRight = 4, cornerRadiusTopLeft = 4)
    .encode(
        x = alt.X("Continent:N", title = "Continent", sort = v8_continent_order,
                axis = alt.Axis(labelAngle = -45, labelFontSize = 10)),
        y = alt.Y("Mean_Math:Q", title = "Mean Math Score",
                scale = alt.Scale(domain = [350, 550], clamp = True)),
        color = alt.Color("Continent:N",
                       scale = alt.Scale(domain = v8_continent_order, range = v8_continent_colors),
                       legend = alt.Legend(title = "Continent", orient = "top", columns = 4)),
        opacity = alt.condition(v8_continent_select, alt.value(1), alt.value(0.4)),
        tooltip = [
            alt.Tooltip("Continent:N", title = "Continent"),
            alt.Tooltip("Mean_Math:Q", title = "Mean Math Score", format = ".0f"),
            alt.Tooltip("Mean_Science:Q", title = "Mean Science Score", format = ".0f"),
            alt.Tooltip("n:Q", title = "Students", format = ",d")
        ]
    )
    .add_params(v8_continent_select)
    .properties(
        name = "view_1",
        title = {"text": "Mathematics Achievement by Geographic Region",
               "subtitle": "Mean plausible values across PISA 2022 participating regions",
               "color": "#FFFFFF", "fontSize": 14, "subtitleColor": "#E0E0E0"},
        width = 350, height = 300
    )
)

v8_right_chart = (
    alt.Chart(v8_gender_subject_long)
    .transform_filter(v8_continent_select)
    .transform_aggregate(
        Score = "mean(Score)",
        Total_Students = "sum(n)",
        groupby = ["Category", "Gender", "Subject"]
    )
    .mark_bar(cornerRadiusTopRight = 4, cornerRadiusBottomRight = 4)
    .encode(
        y = alt.Y("Category:N", title = "Gender × Subject",
                sort = v8_category_order,
                axis = alt.Axis(labelAngle = 0, labelFontSize = 11)),
        x = alt.X("Score:Q", title = "Mean Score",
                scale = alt.Scale(domain = [400, 520], clamp = True)),
        color = alt.Color("Subject:N", title = "Subject",
                       scale = alt.Scale(domain = ["Math", "Science", "Reading"], range = v8_subject_colors),
                       legend = alt.Legend(orient = "top", direction = "horizontal")),
        tooltip = [
            alt.Tooltip("Category:N", title = "Category"),
            alt.Tooltip("Score:Q", title = "Mean Score", format = ".0f"),
            alt.Tooltip("Total_Students:Q", title = "Students", format = ",d")
        ]
    )
    .properties(
        title = {"text": "Academic Performance Across Domains by Gender",
               "subtitle": "Mathematics, science, and reading achievement comparison",
               "color": "#FFFFFF", "fontSize": 14, "subtitleColor": "#E0E0E0"},
        width = 300, height = 280
    )
)

viz8 = alt.hconcat(v8_left_chart, v8_right_chart).resolve_scale(color = "independent").configure_view(
    stroke = None,
    fill = None
).properties(
    background = "transparent"
)
save_chart(viz8, "combined_efficacy_comparison.json")
viz8

## Viz 9: School Belonging, Immigration, and Academic Outcomes

**Left Chart**: Line chart displaying mean academic achievement across domains by school belonging level
**Right Chart**: Scatter plot with regression lines showing mathematics self-efficacy versus persistence by immigration status
**Interaction**: Click on a belonging level to filter the efficacy-persistence scatter plot

### Results Analysis

The visualization reveals a strong positive gradient between school belonging and academic achievement that persists across all three subject domains (mathematics, reading, and science). Students with high belonging scores outperform their low belonging peers by approximately 50 points, representing roughly half a standard deviation on the PISA scale. This achievement premium associated with school connectedness operates through multiple mechanisms including increased engagement with learning activities, stronger relationships with teachers who provide academic support, and reduced likelihood of absenteeism and disciplinary problems that interrupt learning.

The consistency of belonging effects across domains suggests that the psychological benefits of school connectedness translate into generalized academic advantages rather than domain-specific improvements. Students who feel they belong demonstrate better performance in mathematics, reading, and science equally, indicating that belonging operates through broad motivational and engagement pathways rather than subject-specific skill development. This pattern supports intervention approaches that focus on improving school climate and student connectedness rather than domain-specific remediation.

The scatter plot reveals interesting patterns in the relationship between mathematics self-efficacy and persistence across immigration status groups. Native students show a strong positive correlation between these psychological constructs, with confident students also reporting high persistence. First and second generation immigrants display similar positive correlations, but with interesting level differences. First generation immigrants often report higher persistence at comparable efficacy levels, consistent with the immigrant optimism hypothesis that suggests recent immigrants maintain high aspirations despite facing additional challenges.

The regression line slopes appear similar across immigration groups, indicating that the efficacy-persistence relationship operates equivalently regardless of nativity. The variation lies primarily in the intercepts, with immigrant students showing elevated persistence at given efficacy levels. This pattern may reflect selection effects in migration, where families who undertake international moves are characterized by exceptional determination that transmits to their children.

### Encoding Types

The left chart employs connected line marks with overlaid points, enabling viewers to perceive both the continuous trend across domains and discrete values at each point. The line encoding facilitates comparison of slopes across belonging groups, revealing whether achievement gains are uniform or accelerating across subject transitions. Color encoding distinguishes belonging levels while maintaining the ordered relationship through luminance variation.

The right chart combines scatter points with regression line overlays, a layered design that reveals both individual variation and group-level trends. The scatter points show the joint distribution of efficacy and persistence, while the regression lines summarize the linear relationship within each immigration status group. Point opacity is varied by immigration status to reduce overplotting while maintaining visibility of all groups.

### Color Maps

The left chart uses a three-color palette (#E91E63 pink, #00E676 green, #2979FF blue) for belonging levels, selected for maximum perceptual distinctiveness while maintaining visibility against the dark background. The pink-green-blue progression avoids the red-yellow-green traffic light convention that carries evaluative connotations, instead using hue variation to signal categorical differences without implicit value judgments.

The right chart uses blue (#1E88E5) for native students, red (#FF0000) for second generation, and gold (#FFD700) for first generation immigrants. This palette was selected to avoid the conventional blue-for-native association while maintaining high contrast between groups. The warm colors (red, gold) for immigrant groups create visual grouping that facilitates comparison with the cooler native category.

### Data Transformations

The BELONG (school belonging) index is divided into terciles using quantile-based binning, creating three groups of approximately equal size labeled Low, Medium, and High. This transformation converts the continuous composite index into categorical levels suitable for line chart visualization while preserving the ordinal relationship between groups. The tercile approach ensures balanced sample sizes across groups for stable mean estimation.

Achievement data is aggregated by belonging level for each domain, computing group means that summarize performance patterns. The domain variable is extracted from column names and mapped to readable labels for the x-axis display.

The scatter plot data undergoes random sampling to reduce display size to 6,000 observations, preventing overplotting that would obscure distributional patterns. The immigration status variable (IMMIG) is mapped from numeric codes (1, 2, 3) to descriptive labels (Native, Second-Gen, First-Gen) for legend display. The regression transformation fits linear models within each immigration group, extracting trend lines that summarize the efficacy-persistence relationship.

### Interactivity

Clicking on any belonging level line in the left chart filters the right scatter plot to display efficacy-persistence patterns only for students at that belonging level. This linked selection enables investigation of whether the immigration-achievement relationship varies by school connectedness, addressing questions about whether belonging provides differential benefits for immigrant versus native students. Users can explore whether the immigrant persistence advantage persists across belonging levels or is concentrated among students who feel connected to their schools.

In [ ]:
v9_data = pisa_df[
    (pisa_df["BELONG"].notna()) &
    (pisa_df["PV1MATH"].notna()) &
    (pisa_df["PV1READ"].notna()) &
    (pisa_df["PV1SCIE"].notna()) &
    (pisa_df["MATHEFF"].notna()) &
    (pisa_df["MATHPERS"].notna()) &
    (pisa_df["IMMIG"].isin([1, 2, 3, 1.0, 2.0, 3.0]))
].copy()

v9_belong_terciles = v9_data["BELONG"].quantile([0.33, 0.67]).values
v9_data["belong_level"] = pd.cut(
    v9_data["BELONG"],
    bins = [-np.inf, v9_belong_terciles[0], v9_belong_terciles[1], np.inf],
    labels = ["Low Belonging", "Medium Belonging", "High Belonging"]
)

v9_immig_map = {1: "Native", 1.0: "Native", 2: "Second-Gen", 2.0: "Second-Gen", 3: "First-Gen", 3.0: "First-Gen"}
v9_data["immig_status"] = v9_data["IMMIG"].map(v9_immig_map)

v9_belong_order = ["Low Belonging", "Medium Belonging", "High Belonging"]
v9_domain_colors = ["#E91E63", "#00E676", "#2979FF"]
v9_immig_colors = ["#1E88E5", "#FF0000", "#FFD700"]

v9_math = v9_data.groupby("belong_level", observed = True)["PV1MATH"].mean().reset_index()
v9_math["Domain"] = "MATH"
v9_math.columns = ["belong_level", "mean_score", "Domain"]

v9_read = v9_data.groupby("belong_level", observed = True)["PV1READ"].mean().reset_index()
v9_read["Domain"] = "READ"
v9_read.columns = ["belong_level", "mean_score", "Domain"]

v9_scie = v9_data.groupby("belong_level", observed = True)["PV1SCIE"].mean().reset_index()
v9_scie["Domain"] = "SCIE"
v9_scie.columns = ["belong_level", "mean_score", "Domain"]

v9_left_df = pd.concat([v9_math, v9_read, v9_scie], ignore_index = True)

v9_right_df = v9_data[["belong_level", "immig_status", "MATHEFF", "MATHPERS"]].dropna().sample(n = 6000, random_state = 42)

v9_belong_selection = alt.selection_point(fields = ["belong_level"], name = "v9_belong_select")

v9_left_chart = alt.Chart(v9_left_df).mark_line(
    point = alt.OverlayMarkDef(filled = True, size = 80),
    strokeWidth = 4,
    cursor = "pointer"
).encode(
    x = alt.X("Domain:N", title = "Domain",
            sort = ["MATH", "READ", "SCIE"],
            axis = alt.Axis(labelAngle = 0, labelFontSize = 11)),
    y = alt.Y("mean_score:Q", title = "Mean Score",
            scale = alt.Scale(domain = [425, 490])),
    color = alt.Color("belong_level:N", title = "Belonging Level",
                   scale = alt.Scale(domain = v9_belong_order, range = v9_domain_colors),
                   legend = alt.Legend(orient = "top")),
    opacity = alt.condition(v9_belong_selection, alt.value(1), alt.value(0.3)),
    tooltip = [
        alt.Tooltip("Domain:N", title = "Domain"),
        alt.Tooltip("belong_level:N", title = "Belonging Level"),
        alt.Tooltip("mean_score:Q", title = "Mean Score", format = ".1f")
    ]
).add_params(v9_belong_selection).properties(
    title = {"text": "Academic Achievement by School Belonging Level",
           "subtitle": "Performance across mathematics, reading, and science domains",
           "color": "#FFFFFF", "fontSize": 14, "subtitleColor": "#E0E0E0"},
    width = 450, height = 400
)

v9_right_scatter = (
    alt.Chart(v9_right_df)
    .transform_filter(v9_belong_selection)
    .transform_sample(2000)
    .mark_circle(size = 40)
    .encode(
        x = alt.X("MATHEFF:Q", title = "Math Self-Efficacy",
                scale = alt.Scale(domain = [-4, 4])),
        y = alt.Y("MATHPERS:Q", title = "Math Persistence",
                scale = alt.Scale(domain = [-4, 4])),
        color = alt.Color("immig_status:N", title = "Immigration Status",
                       scale = alt.Scale(domain = ["Native", "Second-Gen", "First-Gen"], range = v9_immig_colors),
                       legend = alt.Legend(orient = "top")),
        opacity = alt.Opacity("immig_status:N",
                           scale = alt.Scale(domain = ["Native", "Second-Gen", "First-Gen"], range = [0.4, 0.9, 0.9]),
                           legend = None),
        tooltip = [
            alt.Tooltip("immig_status:N", title = "Immigration"),
            alt.Tooltip("MATHEFF:Q", title = "Self-Efficacy", format = ".2f"),
            alt.Tooltip("MATHPERS:Q", title = "Persistence", format = ".2f")
        ]
    )
)

v9_right_regression = (
    alt.Chart(v9_right_df)
    .transform_filter(v9_belong_selection)
    .transform_regression("MATHEFF", "MATHPERS", groupby = ["immig_status"])
    .mark_line(strokeWidth = 3)
    .encode(
        x = alt.X("MATHEFF:Q"),
        y = alt.Y("MATHPERS:Q"),
        color = alt.Color("immig_status:N", scale = alt.Scale(domain = ["Native", "Second-Gen", "First-Gen"], range = v9_immig_colors))
    )
)

v9_right_chart = alt.layer(v9_right_scatter, v9_right_regression).properties(
    title = {"text": "Mathematics Self-Efficacy and Persistence",
           "subtitle": "Comparing native and immigrant student populations",
           "color": "#FFFFFF", "fontSize": 14, "subtitleColor": "#E0E0E0"},
    width = 400, height = 400
)

viz9 = alt.hconcat(v9_left_chart, v9_right_chart).resolve_scale(color = "independent").configure_view(
    stroke = None,
    fill = None
).properties(
    background = "transparent"
)
save_chart(viz9, "combined_gender_stem.json")
viz9

## Verification

Verify that all JSON output files were generated successfully.

In [ ]:
expected_files = [
    "hsls_math_identity_race.json",
    "combined_immigration.json",
    "pisa_gender_efficacy_dumbbell.json",
    "hsls_gpa_ses_trajectory.json",
    "combined_ses_achievement.json",
    "combined_parent_education.json",
    "pisa_anxiety_performance_heatmap.json",
    "combined_efficacy_comparison.json",
    "combined_gender_stem.json"
]

for filename in expected_files:
    filepath = OUTPUT_DIR / filename
    if filepath.exists():
        size = filepath.stat().st_size